# CropShift Earth v2: a crop-suitability model that learns from all six NASA POWER measurements
**NASA Space Apps 2026 Cairo · "Be An Earth System Trend Detective!" · Lane A (analysis)**

This notebook trains and validates the classical machine-learning engine behind the **Farmland (F2)**, **Crops (F3)** and **Decisions (F4)** tabs, and produces the evidence for the **Methods page (F5)**.

**What is new in v2.** v1 learned crop climates from a small Kaggle table of Indian records (temperature and rainfall only). v2 learns from **where 40 crops are actually grown on Earth today**: IFPRI's MapSPAM 2020 crop map, about 22,000 farmland cells on every continent. Each cell is described by **10 climate features built from all six NASA POWER measurements** in our dataset (temperature, maximum temperature, rainfall, humidity, surface and root-zone soil wetness). The same six measurements, from our own Kaggle dataset, then drive Egypt's maps, trends and +5/+10-year projections.

| Output | Used by | File |
|---|---|---|
| Crop suitability (0 to 1) for 40 crops, rain-fed and irrigated, now / +5 / +10 years with uncertainty | F3 Crops tab, irrigation toggle | `outputs/egypt_crop_suitability.json` |
| Farmland viability ("crop options") per cell with trend + confidence + projection | F2 Farmland tab | same file, `viability` block |
| Ranked investor and planner lists | F4 Decisions tab | `outputs/egypt_decisions.json`, CSVs |
| Runtime engine (numpy only) + packed model for live points anywhere on Earth | `/crops`, `/viability` endpoints | `crop_engine_v2.py`, `models/crop_suitability_v2.npz` |
| Metrics, validation figures, model card | F5 Methods page | `outputs/metrics.json`, `figures/`, `MODEL_CARD.md` |

**Inputs (attach with "Add Input" on Kaggle)**
1. `mohossameldeen/nasa-2026-earth-trends-dataset`: our NASA POWER monthly grid over Egypt, 2001 to 2025 (357 cells, six measurements).
2. **The CropShift v2 training table** (`cropshift_v2_training_cells.parquet`, 7 MB, uploaded as a Kaggle dataset). It holds one row per NASA POWER cell with farming: the 10 climate features (2001 to 2025 average) and the MapSPAM 2020 harvested area of every crop. It is built from two public sources: the NASA POWER archive on AWS and MapSPAM 2020 (IFPRI, CC BY 4.0). If it is not attached and Internet is on, section 2 rebuilds it from those sources in a few minutes.
3. `sayedkhaled2000/egypt-crop-yield-processed-pivot-table` (FAOSTAT Egypt). Used **only** as an independent check. Optional: a fallback copy of the numbers is embedded.

**Run:** attach the inputs, then *Run All*. CPU only. Internet is optional: it is only needed for the live NASA POWER demo (section 12) or to rebuild the training table.

*This data was provided by the International Food Policy Research Institute (IFPRI). IFPRI bears no responsibility for the analyses or interpretations of the data presented here.*

In [ ]:
import os, sys, json, math, time, warnings, platform, shutil, importlib, zipfile, io, urllib.request
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.path import Path as MplPath
from matplotlib.colors import TwoSlopeNorm, ListedColormap
from matplotlib.lines import Line2D

import sklearn
from sklearn.model_selection import GroupKFold, ParameterSampler
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss
import joblib

warnings.filterwarnings('ignore')
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False
try:
    import lightgbm
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

CFG = dict(
    version='crop-suitability-v2',
    seed=42,
    base_year=2025,               # last complete year in the NASA POWER grid
    horizons=[5, 10],             # +5 and +10 years (PROJECT_PLAN F2/F3)
    n_boot=200,                   # bootstrap draws for projection uncertainty
    block_len=3,                  # moving-block length (years)
    farmland_min_ha=1000,         # a cell is farmland if MapSPAM reports >= 1,000 ha harvested (all crops)
    present_share=0.02,           # crop "grown here": >= 2 % of the cell's harvested area
    absent_share=0.002,           # crop "not grown": <= 0.2 %; cells in between are left out as uncertain
    block_deg=10,                 # spatial cross-validation blocks (10 x 10 degrees)
    n_folds=5,
    irrigated_eval_share=0.8,     # "irrigated farmland" test cells: >= 80 % of harvested area irrigated
    rainfed_eval_share=0.2,       # "rain-fed farmland" test cells: <= 20 % irrigated
    suit_threshold=0.5,           # suitable = the climate is at least as typical of cells that grow the crop as of cells that do not
    p_strong=0.01, p_moderate=0.05,
    min_effect_suit=0.02,         # suitability change per decade below this is called "stable"
    min_effect_options=0.10,      # crop-options change per decade below this is called "stable"
    fao_grown_min_ha=10000, fao_absent_max_ha=1500,
    tune_iter=8,                  # random-search candidates per tuned family
    tune_crops=12,                # crops used for tuning (spread over prevalence); the final comparison uses all 40
    max_app_nodes=2_000_000,      # size budget of the app model (all 40 crops): about 20 MB packed, sub-second numpy runtime
    counter_irr_share=0.95,       # rain-fed counter-examples (1): cells with >= 95 % of the harvest irrigated (no rain-fed farming)
    dry_rain_mm=100,              # rain-fed counter-examples (2): unfarmed cells (< farmland_min_ha) with < 100 mm rain a year;
                                  # also the "dry irrigated cells" test set: fully irrigated farmland with < 100 mm rain
)
QUICK = bool(os.environ.get('CROPSHIFT_QUICK'))   # developer smoke test only: fewer families and candidates
if QUICK:
    CFG.update(tune_iter=2, tune_crops=4)
SEED = CFG['seed']
np.random.seed(SEED)
N_JOBS = os.cpu_count() or 4

IS_KAGGLE = Path('/kaggle/working').exists()
OUT = Path('/kaggle/working/cropshift_ml_v2') if IS_KAGGLE else Path('cropshift_ml_v2').resolve()
FIG, MODELS, OUTPUTS = OUT / 'figures', OUT / 'models', OUT / 'outputs'
for d in (FIG, MODELS, OUTPUTS):
    d.mkdir(parents=True, exist_ok=True)

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    pass
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150, 'axes.titleweight': 'bold',
                     'axes.titlesize': 11, 'axes.labelsize': 10, 'legend.fontsize': 8,
                     'xtick.labelsize': 8, 'ytick.labelsize': 8})

def savefig(fig, name):
    fig.savefig(FIG / f'{name}.png', bbox_inches='tight')
    plt.show()

T_START = time.time()
print(f'python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | '
      f'scikit-learn {sklearn.__version__} | xgboost={HAS_XGB} | lightgbm={HAS_LGBM} | cpus={N_JOBS}')
print('Kaggle environment:', IS_KAGGLE, '| outputs ->', OUT)

### Locating the input files
Kaggle mounts datasets under `/kaggle/input/...`, so files are found by name. Locally, set `CROPSHIFT_DATA_DIR`.

In [ ]:
SEARCH_ROOTS = [Path('/kaggle/input'), Path('../input'), Path('data')]
if os.environ.get('CROPSHIFT_DATA_DIR'):
    SEARCH_ROOTS.insert(0, Path(os.environ['CROPSHIFT_DATA_DIR']))

def find_file(filename, kaggle_slug=None, required=True):
    """Find an input file by name under the known roots, else try kagglehub, else fail clearly."""
    for root in SEARCH_ROOTS:
        if root.exists():
            hits = sorted(root.rglob(filename))
            if hits:
                return hits[0]
    if kaggle_slug:
        try:
            import kagglehub
            p = Path(kagglehub.dataset_download(kaggle_slug))
            hits = sorted(p.rglob(filename))
            if hits:
                return hits[0]
        except Exception as e:
            print(f'  kagglehub fallback for {kaggle_slug} failed: {type(e).__name__}: {e}')
    if required:
        raise FileNotFoundError(f'{filename} not found. Attach the Kaggle dataset "{kaggle_slug}" with "Add Input".')
    return None

TRAIN_PARQUET = find_file('cropshift_v2_training_cells.parquet', None, required=False)
POWER_PARQUET = find_file('power_monthly.parquet', None, required=False)
FAO_CSV = find_file('FAOSTAT_Egypt_Agriculture_Raw.csv', 'sayedkhaled2000/egypt-crop-yield-processed-pivot-table', required=False)
print('training table:', TRAIN_PARQUET or '(not attached: will be rebuilt from NASA POWER on AWS + MapSPAM 2020, needs Internet)')
print('NASA POWER    :', POWER_PARQUET or '(parquet not found, will use the JSON files)')
print('FAOSTAT       :', FAO_CSV or '(not attached, using embedded fallback values)')

## 1. Why this training data
A crop-suitability model needs **labels**: places where a crop is grown and places where it is not. Our NASA dataset has the climate but no crop labels, so it cannot train a model on its own.

* **v1** used `atharvaingle/crop-recommendation-dataset` (2,200 synthetic-looking records from India). Only temperature and rainfall could be used, India's climates set the envelopes, and wheat, potato and tomato were missing.
* **v2** uses **MapSPAM 2020** (IFPRI, Harvard Dataverse doi:10.7910/DVN/SWPENT, CC BY 4.0). It estimates the harvested area of 46 crops on a 10 km grid worldwide, split into irrigated and rain-fed farming, and is reconciled with national and sub-national statistics (about 2019 to 2021). We sum it onto the NASA POWER grid (0.5° x 0.625°, the same cells as our Egypt dataset). Each cell then gets the same 10 features we compute for Egypt, from the same six NASA measurements over 2001 to 2025.

MapSPAM is not on Kaggle. We therefore publish the compact training table as a Kaggle dataset, and this notebook can rebuild it from the original sources.

**Crops.** 40 of the 46 MapSPAM crops. The six vague "other ..." groups are dropped. Olive and date palm are not separate MapSPAM crops, so they are still not covered.

In [ ]:
CROP_NAMES = {
    'whea': 'wheat', 'rice': 'rice', 'maiz': 'maize', 'barl': 'barley', 'pmil': 'pearl millet', 'mill': 'small millets',
    'sorg': 'sorghum', 'pota': 'potato', 'swpo': 'sweet potato', 'yams': 'yams', 'cass': 'cassava', 'bean': 'beans',
    'chic': 'chickpea', 'cowp': 'cowpea', 'pige': 'pigeon pea', 'lent': 'lentil', 'soyb': 'soybean', 'grou': 'groundnut',
    'cnut': 'coconut', 'oilp': 'oil palm', 'sunf': 'sunflower', 'rape': 'rapeseed', 'sesa': 'sesame', 'sugc': 'sugarcane',
    'sugb': 'sugar beet', 'cott': 'cotton', 'coff': 'arabica coffee', 'rcof': 'robusta coffee', 'coco': 'cocoa',
    'teas': 'tea', 'toba': 'tobacco', 'bana': 'banana', 'plnt': 'plantain', 'citr': 'citrus', 'trof': 'tropical fruit',
    'temf': 'temperate fruit', 'toma': 'tomato', 'onio': 'onion', 'vege': 'vegetables', 'rubb': 'rubber'}
SPAM_ALL = ['whea', 'rice', 'maiz', 'barl', 'mill', 'sorg', 'ocer', 'pota', 'swpo', 'yams', 'cass', 'orts', 'bean', 'chic', 'cowp',
            'pige', 'lent', 'opul', 'soyb', 'grou', 'cnut', 'oilp', 'sunf', 'rape', 'sesa', 'ooil', 'sugc', 'sugb', 'cott', 'ofib',
            'coff', 'coco', 'teas', 'toba', 'bana', 'plnt', 'trof', 'temf', 'vege', 'rest', 'pmil', 'rcof', 'citr', 'toma', 'onio', 'rubb']
CODES = list(CROP_NAMES)
CROPS = [CROP_NAMES[c] for c in CODES]
print(f'{len(CROPS)} crops:', ', '.join(CROPS))

## 2. From six NASA measurements to ten climate features
Every feature is computed from one year of monthly NASA POWER values. The same function is used for the global training table, for our Egypt grid and for live points in the API.

| Feature | NASA measurement | Meaning |
|---|---|---|
| `t_mean`, `t_coldest`, `t_warmest` | T2M | annual mean, coldest-month and warmest-month temperature (°C) |
| `tmax_hottest` | T2M_MAX | highest monthly maximum temperature (heat stress, °C) |
| `rain_annual`, `rain_wettest3_share` | PRECTOTCORR | annual rainfall (mm) and share falling in the wettest 3 months (seasonality) |
| `rh_mean` | RH2M | annual mean relative humidity (%) |
| `soil_top_wetness` | GWETTOP | annual mean surface soil wetness (0 to 1) |
| `soil_root_wetness`, `soil_root_wetness_min` | GWETROOT | annual mean and driest-month root-zone soil wetness (0 to 1) |

The model is trained on the **2001 to 2025 average** of each feature, the climate a crop map reflects. Projections apply the same model to the trend-line values for now, +5 and +10 years.

In [ ]:
FEATURES = ['t_mean', 't_coldest', 't_warmest', 'tmax_hottest', 'rain_annual', 'rain_wettest3_share',
            'rh_mean', 'soil_top_wetness', 'soil_root_wetness', 'soil_root_wetness_min']
FEATURE_PARAM = {'t_mean': 'T2M', 't_coldest': 'T2M', 't_warmest': 'T2M', 'tmax_hottest': 'T2M_MAX', 'rain_annual': 'PRECTOTCORR',
                 'rain_wettest3_share': 'PRECTOTCORR', 'rh_mean': 'RH2M', 'soil_top_wetness': 'GWETTOP',
                 'soil_root_wetness': 'GWETROOT', 'soil_root_wetness_min': 'GWETROOT'}
POWER_VARS = ['T2M', 'T2M_MAX', 'PRECTOTCORR', 'RH2M', 'GWETTOP', 'GWETROOT']
DAYS = np.array([31, 28.25, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])

def annual_features(T, TX, P, RH, GT, GR):
    """Each argument: array (..., 12) of monthly values for one year. Returns (..., 10) in FEATURES order."""
    T, TX, P, RH, GT, GR = (np.asarray(a, dtype=np.float64) for a in (T, TX, P, RH, GT, GR))
    rain_m = P * DAYS                                            # mm/day -> mm per month
    rain_y = rain_m.sum(-1)
    ext = np.concatenate([rain_m, rain_m[..., :2]], axis=-1)     # wrap around the year end
    q3 = ext[..., 0:12] + ext[..., 1:13] + ext[..., 2:14]
    share = np.where(rain_y > 1.0, q3.max(-1) / np.maximum(rain_y, 1e-9), 1.0)
    return np.stack([T.mean(-1), T.min(-1), T.max(-1), TX.max(-1), rain_y, share,
                     RH.mean(-1), GT.mean(-1), GR.mean(-1), GR.min(-1)], axis=-1)

### Training table: load it, or rebuild it from the original sources
The rebuild downloads MapSPAM 2020 harvested area (about 110 MB) and the six NASA POWER variables for 2001 to 2025 between 60°S and 75°N from the official NASA POWER archive on AWS (Zarr format). It then sums MapSPAM onto the POWER grid and computes the features. This is exactly how the attached table was made.

In [ ]:
SPAM_URL = ('https://www.dropbox.com/scl/fi/py1g3yovxt8ws7hcfq3ys/spam2020V2r2_global_harvested_area.csv.zip'
            '?rlkey=6oay3zyojlhcz1416qsf10o31&st=2dkfuh8d&dl=1')
POWER_ZARR = 'https://nasa-power.s3.us-west-2.amazonaws.com/merra2/temporal/power_merra2_monthly_temporal_utc.zarr'

def build_training_table(dest):
    import subprocess
    try:
        import xarray as xr, fsspec, zarr  # noqa: F401
    except Exception:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'xarray', 'zarr', 'fsspec', 'aiohttp'], check=True)
        import xarray as xr, fsspec
    t0 = time.time()
    zpath = OUT.parent / 'spam2020V2r2_global_harvested_area.csv.zip'      # outside OUT, so it is not zipped
    if not zpath.exists():
        urllib.request.urlretrieve(SPAM_URL, zpath)
    print(f'MapSPAM downloaded ({zpath.stat().st_size / 1e6:.0f} MB, {time.time() - t0:.0f} s)')
    spam = None
    with zipfile.ZipFile(zpath) as z:
        for tech in ['TA', 'TI', 'TR']:
            name = [n for n in z.namelist() if n.endswith(f'_H_{tech}.csv')][0]
            df = pd.read_csv(z.open(name), usecols=['x', 'y', 'ADM0_NAME'] + SPAM_ALL,
                             dtype={c: 'float32' for c in SPAM_ALL} | {'x': 'float64', 'y': 'float64', 'ADM0_NAME': 'str'})
            df['ilat'] = np.rint((df['y'].to_numpy() + 90.0) / 0.5).astype(int)
            df['ilon'] = np.rint((df['x'].to_numpy() + 180.0) / 0.625).astype(int) % 576
            agg = df.groupby(['ilat', 'ilon'])[SPAM_ALL].sum().add_prefix(f'{tech}_')
            if tech == 'TA':
                df['tot'] = df[SPAM_ALL].sum(axis=1)
                ctry = (df.groupby(['ilat', 'ilon', 'ADM0_NAME'])['tot'].sum().reset_index().sort_values('tot')
                          .drop_duplicates(['ilat', 'ilon'], keep='last').set_index(['ilat', 'ilon'])['ADM0_NAME'])
                agg = agg.join(ctry.rename('country'))
            spam = agg if spam is None else spam.join(agg, how='outer')
    spam = spam.reset_index()
    num = [c for c in spam.columns if c[:3] in ('TA_', 'TI_', 'TR_')]
    spam[num] = spam[num].fillna(0).astype('float32')
    spam['lat'], spam['lon'] = -90.0 + 0.5 * spam['ilat'], -180.0 + 0.625 * spam['ilon']
    R = 6371.0
    spam['cell_area_ha'] = (R ** 2 * np.radians(0.625) * (np.sin(np.radians(spam['lat'] + 0.25)) - np.sin(np.radians(spam['lat'] - 0.25))) * 100).astype('float32')
    for tech in ['TA', 'TI', 'TR']:
        spam[f'{tech}_total'] = spam[[f'{tech}_{c}' for c in SPAM_ALL]].sum(axis=1)
    print(f'MapSPAM on the POWER grid: {len(spam)} cells ({time.time() - t0:.0f} s)')
    ds = xr.open_zarr(fsspec.get_mapper(POWER_ZARR), consolidated=True)
    ds = ds[POWER_VARS].sel(time=slice('2001-01-01', '2025-12-31'), lat=slice(-60, 75))
    lat, lon = ds.lat.values, ds.lon.values
    A = {}
    for v in POWER_VARS:
        A[v] = np.concatenate([ds[v].isel(lat=slice(k, k + 30)).load().values.astype(np.float32) for k in range(0, len(lat), 30)], axis=1)
        print(f'  NASA POWER {v} {A[v].shape} ({time.time() - t0:.0f} s)')
    nY = A['T2M'].shape[0] // 12
    yearly = lambda a: np.transpose(a[:nY * 12].reshape(nY, 12, a.shape[1], a.shape[2]), (2, 3, 0, 1))
    clim = annual_features(*(yearly(A[v]) for v in POWER_VARS)).mean(axis=2)
    i0 = int(round((lat[0] + 90) / 0.5))
    spam = spam[(spam.ilat >= i0) & (spam.ilat < i0 + len(lat))].copy()
    feat = pd.DataFrame(clim[spam.ilat.to_numpy() - i0, spam.ilon.to_numpy()], columns=FEATURES, index=spam.index)
    tab = pd.concat([spam[['ilat', 'ilon', 'lat', 'lon', 'country', 'cell_area_ha']], feat,
                     spam[[c for c in spam.columns if c[:3] in ('TA_', 'TI_', 'TR_')]]], axis=1)
    tab = tab[np.isfinite(tab[FEATURES]).all(axis=1)].reset_index(drop=True)
    num = tab.select_dtypes('float64').columns
    tab[num] = tab[num].astype('float32')
    tab.to_parquet(dest, index=False)
    print(f'training table rebuilt: {len(tab)} cells -> {dest} ({time.time() - t0:.0f} s)')
    return dest

if TRAIN_PARQUET is None:
    TRAIN_PARQUET = build_training_table(OUT / 'cropshift_v2_training_cells.parquet')
cells_all = pd.read_parquet(TRAIN_PARQUET)
print(f'{len(cells_all):,} NASA POWER cells with farming | world harvested area {cells_all.TA_total.sum() / 1e6:,.0f} Mha '
      f'({cells_all.TI_total.sum() / cells_all.TA_total.sum():.0%} irrigated)')

## 3. Labels: where each crop is grown and where it is not
* **Farmland cell:** MapSPAM reports at least 1,000 ha harvested (all crops).
* **Crop present:** the crop covers at least **2 %** of the cell's harvested area. **Absent:** at most **0.2 %**. Cells in between are left out as uncertain.
* **Irrigation share:** irrigated harvested area divided by total harvested area. It becomes a model **input**. With it, the same model answers both app modes: irrigated (share = 1) and rain-fed (share = 0). The app's irrigation slider can use any value in between.

Shares, not hectares, decide presence. A narrow Nile-valley cell holds far less farmland than a cell in the US corn belt. A fixed hectare rule would mark most crops as absent in small-farmland cells and bias the model against desert agriculture. Section 5 tests both rules.

In [ ]:
farm = cells_all[cells_all.TA_total >= CFG['farmland_min_ha']].reset_index(drop=True)
farm['irr_share'] = (farm.TI_total / farm.TA_total).clip(0, 1).astype(float)

def share_labels(tab, prefix='TA_', pres=CFG['present_share'], absn=CFG['absent_share']):
    out = {}
    tot = np.maximum(tab['TA_total'].to_numpy(float), 1.0)
    for code, name in CROP_NAMES.items():
        s = tab[prefix + code].to_numpy(float) / tot
        out[name] = np.where(s >= pres, 1.0, np.where(s <= absn, 0.0, np.nan))
    return pd.DataFrame(out, index=tab.index)

def hectare_labels(tab, prefix='TA_', pres=1000.0, absn=100.0):
    out = {}
    for code, name in CROP_NAMES.items():
        a = tab[prefix + code].to_numpy(float)
        out[name] = np.where(a >= pres, 1.0, np.where(a <= absn, 0.0, np.nan))
    return pd.DataFrame(out, index=tab.index)

LAB = share_labels(farm)
COLS = FEATURES + ['irr_share']
X_ALL = farm[COLS].to_numpy(float)
GROUPS = (np.floor((farm.lat + 90) / CFG['block_deg']) * 1000 + np.floor((farm.lon + 180) / CFG['block_deg'])).astype(int).to_numpy()
FOLDS = list(GroupKFold(n_splits=CFG['n_folds']).split(X_ALL, groups=GROUPS))
EGYPT = (farm.country == 'Egypt').to_numpy()
IRR_DOM = (farm.irr_share >= CFG['irrigated_eval_share']).to_numpy()
RF_DOM = (farm.irr_share <= CFG['rainfed_eval_share']).to_numpy()
DRY_IRR = ((farm.irr_share >= CFG['counter_irr_share']) & (farm.rain_annual < CFG['dry_rain_mm'])).to_numpy()
# unfarmed deserts (rain-fed counter-examples, type 2), assigned to the same spatial folds by block
desert = cells_all[(cells_all.TA_total < CFG['farmland_min_ha']) & (cells_all.rain_annual < CFG['dry_rain_mm'])].reset_index(drop=True)
desert['irr_share'] = 0.0                                   # counter-examples are used in rain-fed form only
X_DESERT = np.column_stack([desert[FEATURES].to_numpy(float), np.zeros(len(desert))])
_g_desert = (np.floor((desert.lat + 90) / CFG['block_deg']) * 1000 + np.floor((desert.lon + 180) / CFG['block_deg'])).astype(int).to_numpy()
_fold_of = {g: k for k, (_, te) in enumerate(FOLDS) for g in np.unique(GROUPS[te])}
DESERT_FOLD = np.array([_fold_of.get(g, g % CFG['n_folds']) for g in _g_desert])
DESERT_EGYPT = desert.country.eq('Egypt').to_numpy()
lab_summary = pd.DataFrame({'present cells': (LAB == 1).sum(), 'absent cells': (LAB == 0).sum(), 'uncertain (left out)': LAB.isna().sum(),
                            'present in Egypt': (LAB[EGYPT] == 1).sum(),
                            'world harvested area (Mha)': [farm['TA_' + c].sum() / 1e6 for c in CODES]}, index=CROPS)
print(f'{len(farm):,} farmland cells in {len(np.unique(GROUPS))} spatial blocks | Egypt: {EGYPT.sum()} farmland cells | '
      f'irrigated farmland (>= 80 %): {IRR_DOM.sum():,} | rain-fed farmland (<= 20 %): {RF_DOM.sum():,} | '
      f'fully irrigated (>= 95 %): {(farm.irr_share >= CFG["counter_irr_share"]).sum():,}, of which dry (< 100 mm rain): {DRY_IRR.sum():,} | '
      f'unfarmed deserts (< 100 mm rain, < 1,000 ha farmed): {len(desert):,}')
display(lab_summary.sort_values('world harvested area (Mha)', ascending=False).round(1))

In [ ]:
GRID_EXTENT = (-180 - 0.3125, 180 - 0.3125, -90 - 0.25, 90 + 0.25)   # NASA POWER cell edges (0.5 x 0.625 degrees)

def grid_image(df, values):
    """Place per-cell values on the global NASA POWER grid (361 x 576) for imshow; empty cells are NaN."""
    img = np.full((361, 576), np.nan)
    img[df['ilat'].to_numpy(int), df['ilon'].to_numpy(int)] = np.asarray(values, float)
    return img

def world_axes(ax, title):
    ax.set_xlim(-180, 180); ax.set_ylim(-58, 72); ax.set_aspect('equal')
    ax.set_title(title); ax.set_xlabel('longitude'); ax.set_ylabel('latitude')

fig, axes = plt.subplots(2, 1, figsize=(12.5, 10.5))
other = cells_all[cells_all.TA_total < CFG['farmland_min_ha']]
for ax, col, cmap, title, lab in [
        (axes[0], farm['irr_share'], 'viridis_r', 'A. Farmland cells and their irrigated share', 'irrigated share of harvested area'),
        (axes[1], (LAB == 1).sum(axis=1), 'YlGn', 'B. Number of the 40 crops grown in each cell', 'crops present\n(>= 2 % of harvested area)')]:
    ax.imshow(grid_image(other, np.ones(len(other))), extent=GRID_EXTENT, origin='lower', cmap=ListedColormap(['#d9d9d9']),
              interpolation='nearest')
    m = ax.imshow(grid_image(farm, col), extent=GRID_EXTENT, origin='lower', cmap=cmap, interpolation='nearest')
    ax.imshow(grid_image(farm[EGYPT], np.ones(int(EGYPT.sum()))), extent=GRID_EXTENT, origin='lower',
              cmap=ListedColormap(['red']), interpolation='nearest')
    world_axes(ax, title)
    plt.colorbar(m, ax=ax, shrink=.8, pad=.015, label=lab)
axes[0].legend(handles=[Line2D([], [], marker='s', ls='', c='red', ms=6, label='Egypt farmland cells'),
                        Line2D([], [], marker='s', ls='', c='#bdbdbd', ms=6, label='cells with < 1,000 ha farmed')],
               loc='lower left', frameon=True)
fig.suptitle(f'Figure 1. Training data: {len(farm):,} farmland cells on the NASA POWER grid (MapSPAM 2020)', fontweight='bold')
fig.tight_layout()
savefig(fig, 'fig01_training_data_map')

## 4. Our NASA dataset: Egypt, 2001 to 2025
`power_monthly.parquet` is a tidy long table (`cell_id, lat, lon, time, variable, value`). If it is missing, the six original NASA POWER JSON files are parsed instead. We compute the same 10 features per cell and year. The first check: **do our files give the same features as the training table for the same cells?** If yes, the model learns and is applied on one consistent scale.

In [ ]:
if POWER_PARQUET is not None:
    power = pd.read_parquet(POWER_PARQUET)
else:
    recs = []
    for v in POWER_VARS:
        f = find_file(f'{v}_2001_2025.json', 'mohossameldeen/nasa-2026-earth-trends-dataset')
        js = json.load(open(f))
        fill = js.get('header', {}).get('fill_value', -999.0)
        for feat_ in js['features']:
            lon_, lat_ = feat_['geometry']['coordinates'][:2]
            for ym, val in feat_['properties']['parameter'][v].items():
                if ym[4:] == '13':
                    continue
                recs.append((f'p_{float(lat_):.1f}_{float(lon_):.1f}', float(lat_), float(lon_), f'{ym[:4]}-{ym[4:]}', v,
                             np.nan if val == fill else val))
    power = pd.DataFrame(recs, columns=['cell_id', 'lat', 'lon', 'time', 'variable', 'value'])
power['value'] = power['value'].where(power['value'] > -900)
wide = power.pivot_table(index=['cell_id', 'lat', 'lon', 'time'], columns='variable', values='value').reset_index()
wide.columns.name = None
wide['year'] = wide['time'].str[:4].astype(int)
wide['month'] = wide['time'].str[5:7].astype(int)
wide = wide.sort_values(['cell_id', 'year', 'month']).reset_index(drop=True)
print(f"{wide.cell_id.nunique()} cells | {wide.time.min()} to {wide.time.max()} | variables: {[v for v in POWER_VARS if v in wide.columns]}")

cells = wide.groupby(['cell_id', 'lat', 'lon']).agg(gwetroot_mean=('GWETROOT', 'mean')).reset_index().sort_values(['lat', 'lon']).reset_index(drop=True)
cells['is_land'] = cells['gwetroot_mean'].notna()
EGYPT_POLY = [(24.9, 22.0), (36.9, 22.0), (35.6, 23.1), (35.6, 23.9), (34.9, 25.07), (34.28, 26.1), (33.94, 26.75),
              (33.82, 27.25), (33.08, 28.36), (32.65, 29.11), (32.55, 29.97), (32.72, 29.6), (33.1, 29.05),
              (33.62, 28.24), (34.25, 27.73), (34.33, 27.92), (34.52, 28.5), (34.67, 29.03), (34.9, 29.49),
              (34.22, 31.32), (33.8, 31.13), (32.3, 31.27), (31.84, 31.52), (31.1, 31.6), (30.37, 31.47),
              (30.06, 31.32), (29.9, 31.2), (28.95, 30.83), (27.23, 31.37), (25.15, 31.57), (24.9, 31.6)]
cells['in_egypt'] = MplPath(EGYPT_POLY).contains_points(cells[['lon', 'lat']].to_numpy())
# Farmland flag from MapSPAM (replaces v1's hand-drawn Nile cell list): Egyptian cells with >= 1,000 ha harvested.
key = lambda la, lo: (np.round(np.asarray(la, float), 3), np.round(np.asarray(lo, float), 3))
spam_idx = cells_all.set_index(pd.MultiIndex.from_arrays(key(cells_all.lat, cells_all.lon)))
m_ = spam_idx.reindex(pd.MultiIndex.from_arrays(key(cells.lat, cells.lon)))
cells['farmland_ha'] = m_['TA_total'].fillna(0).to_numpy()
cells['irrigated_share'] = (m_['TI_total'] / m_['TA_total']).to_numpy()
cells['nile_farmland'] = cells['is_land'] & cells['in_egypt'] & (cells['farmland_ha'] >= CFG['farmland_min_ha'])
print(cells[['is_land', 'in_egypt', 'nile_farmland']].sum().to_string())

# yearly features per cell (complete years only)
YEARS = np.sort(wide.year.unique())
cube = {v: wide.pivot_table(index=['cell_id', 'year'], columns='month', values=v, dropna=False) for v in POWER_VARS}
idx = cube['T2M'].index
F_rows = annual_features(*(cube[v].reindex(idx).to_numpy() for v in POWER_VARS))
annual = pd.DataFrame(F_rows, columns=FEATURES, index=idx).reset_index()
annual = annual.merge(cells[['cell_id', 'lat', 'lon', 'is_land', 'in_egypt', 'nile_farmland']], on='cell_id')
annual['complete'] = np.isfinite(annual[FEATURES]).all(axis=1)
print(f'{annual.complete.sum():,} complete cell-years | years {YEARS[0]} to {YEARS[-1]}')

# consistency check against the training table (same NASA cells, 2001-2025 mean)
clim_user = annual[annual.complete].groupby(['lat', 'lon'])[FEATURES].mean().reset_index()
chk = clim_user.merge(cells_all[['lat', 'lon'] + FEATURES].assign(lat=lambda d: d.lat.round(3), lon=lambda d: d.lon.round(3)),
                      left_on=[clim_user.lat.round(3), clim_user.lon.round(3)], right_on=['lat', 'lon'], suffixes=('_ours', '_train'))
consistency = pd.DataFrame({f: {'NASA parameter': FEATURE_PARAM[f], 'mean |difference|': (chk[f + '_ours'] - chk[f + '_train']).abs().mean(),
                               'max |difference|': (chk[f + '_ours'] - chk[f + '_train']).abs().max(),
                               'typical value in Egypt': chk[f + '_train'].mean()} for f in FEATURES}).T
print(f'Cells present in both our Egypt dataset and the global training table: {len(chk)}')
display(consistency.round(4))

The differences are tiny: NASA POWER occasionally revises a month between data releases. Our Kaggle dataset and the global training table are therefore **the same measurements on the same grid**.

### Domain check: where does Egypt sit among the world's farmland?
The model can only be trusted inside the range of climates it learned from. Figure 2 places Egypt's farmland on top of the world's.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.9))
pairs = [('t_mean', 'rain_annual', 'annual mean temperature (°C)', 'annual rainfall (mm)'),
         ('tmax_hottest', 'rh_mean', 'hottest monthly maximum temperature (°C)', 'mean relative humidity (%)'),
         ('t_coldest', 'soil_root_wetness', 'coldest-month temperature (°C)', 'root-zone soil wetness (0-1)')]
for ax, (fx, fy, lx, ly) in zip(axes, pairs):
    m = ax.scatter(farm[fx], farm[fy], s=1.2, c=farm['irr_share'], cmap='viridis_r', alpha=.5, linewidths=0)
    e = farm[EGYPT]
    ax.scatter(e[fx], e[fy], s=10, facecolors='none', edgecolors='red', linewidths=.8)
    ax.set_xlabel(lx); ax.set_ylabel(ly)
    if fy == 'rain_annual':
        ax.set_yscale('symlog', linthresh=50); ax.set_ylim(0, 8000)
axes[0].set_title('A. Egypt farms with almost no rain')
axes[1].set_title('B. Hot and dry air, like other irrigated deserts')
axes[2].set_title('C. Root zone as dry as any farmland on Earth')
plt.colorbar(m, ax=axes, shrink=.8, label='irrigated share of harvested area')
axes[0].legend(handles=[Line2D([], [], marker='o', ls='', mfc='none', mec='red', label='Egypt farmland cells')], loc='upper left', frameon=True)
fig.suptitle('Figure 2. Egypt inside the training climate space (each dot is one farmland cell, 2001-2025 average)', fontweight='bold', y=1.02)
savefig(fig, 'fig02_domain_check')
P01, P99 = farm[FEATURES].quantile(0.01).to_numpy(), farm[FEATURES].quantile(0.99).to_numpy()
eg_in = ((farm.loc[EGYPT, FEATURES] >= P01) & (farm.loc[EGYPT, FEATURES] <= P99)).mean()
print('Share of Egypt farmland cells inside the 1st-99th percentile range of world farmland, per feature:')
print(eg_in.round(2).to_string())

Egypt sits at the dry, hot edge of the world's farmland, together with other irrigated deserts (dark purple points: Iraq, Pakistan's Sindh, Central Asia, Arizona and California, coastal Peru). NASA's soil-wetness fields come from a land model that does not simulate irrigation. They therefore describe the **natural** dryness of the land, not the watered field. This is why irrigation has to be an explicit model input.

### Data check: is every trend a real trend?
A trend detective also checks the instrument. For each of the six NASA measurements we average Egypt's farmland cells per year and run the **Pettitt test** for one abrupt shift (Pettitt 1979). A gradual trend also produces a "shift", so the size of the jump matters. A measurement is flagged as a **possible data break** when the shift is significant (p < 0.05) and the mean after it differs from the mean before by more than 50 %. A sudden step can be real (a run of wet years) or a change in the data product. Either way, a straight trend line through a step overstates the long-term trend, and extending it to +10 years overstates it further.

In [ ]:
def pettitt(x):
    """Pettitt (1979) change-point test. Returns (index of the last value before the shift, approximate p-value)."""
    x = np.asarray(x, float); n = len(x)
    s = np.sign(x[None, :] - x[:, None])                         # s[i, j] = sign(x_j - x_i)
    U = np.array([s[:t + 1, t + 1:].sum() for t in range(n - 1)])
    t = int(np.argmax(np.abs(U))); K = abs(U[t])
    return t, float(min(1.0, 2 * np.exp(-6 * K ** 2 / (n ** 3 + n ** 2))))

BREAK_SERIES = {'t_mean': 'T2M: annual mean temperature (°C)', 'tmax_hottest': 'T2M_MAX: hottest monthly maximum (°C)',
                'rain_annual': 'PRECTOTCORR: annual rainfall (mm)', 'rh_mean': 'RH2M: mean relative humidity (%)',
                'soil_top_wetness': 'GWETTOP: surface soil wetness (0-1)', 'soil_root_wetness': 'GWETROOT: root-zone soil wetness (0-1)'}
eg_year = annual[annual.complete & annual.nile_farmland].groupby('year')[list(BREAK_SERIES)].mean()
brk = {}
for f, lab_ in BREAK_SERIES.items():
    t, p = pettitt(eg_year[f].to_numpy())
    before, after = eg_year[f].iloc[:t + 1].mean(), eg_year[f].iloc[t + 1:].mean()
    rel = after / before - 1 if before else np.nan
    brk[f] = {'NASA measurement': lab_, 'shift starts': int(eg_year.index[t + 1]), 'Pettitt p': p, 'mean before': before,
              'mean after': after, 'relative change': rel, 'possible data break': bool(p < 0.05 and abs(rel) > 0.5)}
DATA_BREAKS = pd.DataFrame(brk).T
display(DATA_BREAKS)
def _p_txt(p):
    return 'p < 0.001' if p < 0.001 else f'p = {p:.3f}'

DATA_NOTES = [f"NASA POWER {r['NASA measurement'].split(':')[0]} over Egypt's farmland steps up by {r['relative change']:+.0%} from "
              f"{r['shift starts']} (mean {r['mean before']:.3g} before, {r['mean after']:.3g} after; Pettitt test {_p_txt(r['Pettitt p'])}). "
              f"The step may be real or a change in the data product; we could not confirm which. A straight trend through a step "
              f"overstates the long-term trend, so treat this measurement's trend and the projections that extend it with caution."
              + (" NASA's soil wetness, which responds to rainfall, rises at the same time." if f == 'rain_annual' else '')
              for f, r in DATA_BREAKS.iterrows() if r['possible data break']]
print('\n'.join(DATA_NOTES) or 'No possible data break found.')
# the same flags go into the model file, so the app shows the note with every report inside Egypt
DATA_FLAGS = [{'nasa_parameter': FEATURE_PARAM[f], 'features': [g for g in FEATURES if FEATURE_PARAM[g] == FEATURE_PARAM[f]],
               'from_year': int(r['shift starts']), 'region': 'Egypt', 'polygon': [[float(x), float(y)] for x, y in EGYPT_POLY], 'note': note}
              for (f, r), note in zip(DATA_BREAKS[DATA_BREAKS['possible data break'].astype(bool)].iterrows(), DATA_NOTES)]

fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharex=True)
for ax, (f, lab_) in zip(axes.ravel(), BREAK_SERIES.items()):
    r = DATA_BREAKS.loc[f]
    flag = bool(r['possible data break'])
    ax.plot(eg_year.index, eg_year[f], 'o-', c='#cb181d' if flag else '#08519c', ms=4)
    yb = r['shift starts']
    ax.hlines([r['mean before'], r['mean after']], [eg_year.index[0], yb], [yb - 1, eg_year.index[-1]], colors='k', linestyles='--', lw=1)
    ax.axvline(yb - 0.5, c='grey', lw=.8, ls=':')
    ax.set_title(lab_, fontsize=9.5)
    ax.text(.02, .95, f"shift from {yb}: {r['relative change']:+.0%} ({_p_txt(r['Pettitt p'])})" + ('\nPOSSIBLE DATA BREAK' if flag else ''),
            transform=ax.transAxes, va='top', fontsize=8, color='#cb181d' if flag else 'k')
fig.suptitle("Figure 2b. Egypt's farmland, yearly means of the six NASA measurements, with the Pettitt shift test (dashed = mean before / after)",
             fontweight='bold')
fig.tight_layout()
savefig(fig, 'fig02b_data_breaks')

In our dataset the test flags rainfall (PRECTOTCORR) only (table above). Surface soil wetness, which responds to rain, rises at the same time but stays under the 50 % threshold; temperature and humidity change by a few percent. NASA POWER's meteorology comes from the MERRA-2 reanalysis. We have not identified the cause of the step and do not claim one. What matters for the app: the **rainfall trend** over Egypt mixes any real change with this step, and the +5 / +10 projections extend that trend. The data note (`data_quality_notes` in the exported JSON) is therefore shown next to rainfall trends and rain-fed projections. The model itself is trained on 2001 to 2025 averages, which a step shifts much less than it shifts a trend line.

## 5. Model design: how should the irrigation toggle work?
Before choosing an algorithm we choose **what to learn**. Five designs, plus one ablation, are compared with one fixed learner (LightGBM). Each is scored on tests that never see the data they are scored on:
* **Spatial cross-validation:** the world is cut into 10° x 10° blocks and whole blocks are held out (5 folds), so the model is always tested on regions it did not see.
* **Irrigated farmland** (≥ 80 % irrigated) is scored in irrigated mode; **rain-fed farmland** (≤ 20 % irrigated) in rain-fed mode.
* **Leave-Egypt-out:** the model is retrained without any Egyptian cell. It is then scored on Egypt's 116 farmland cells (MapSPAM) and against FAOSTAT's national list of crops Egypt grows at scale (≥ 10,000 ha) or not (≤ 1,500 ha).
* **Rain-fed mode in deserts:** in held-out farmland that is fully irrigated (≥ 95 %) and gets less than 100 mm of rain a year, MapSPAM shows no rain-fed farming at all. So in rain-fed mode, every crop there should be rated unsuitable. We count how often a design wrongly calls a crop suitable there. The same count is made on held-out **unfarmed deserts** (< 100 mm of rain, < 1,000 ha farmed).

**Rain-fed counter-examples.** A trap hides in design C. In the training data, a desert climate always comes with irrigation share ≈ 1. The model never sees "desert climate + no irrigation", so in rain-fed mode a tree model simply extrapolates. It can then call wheat "suitable" for rain-fed farming in Cairo, where it rains 25 mm a year. MapSPAM itself also lists some rain-fed sorghum, sesame and millet in places with less than 100 mm of rain a year (probably flood-recession or spate farming, which lives on river or runoff water rather than rain), and the model copies that to other deserts. The fix uses the crop map's own evidence, in two kinds of counter-examples, all with irrigation share 0 and the label "not grown":

1. **Irrigated copies:** where ≥ 95 % of the harvest is irrigated, rain-fed farming of any crop is absent. Each such cell is added once more in rain-fed form.
2. **Unfarmed deserts:** cells with less than 100 mm of rain where MapSPAM finds less than 1,000 ha farmed (about 1,100 cells, mostly in the Sahara, Arabia, western China, Iran and the Atacama).

Counter-examples come only from the training folds (deserts are assigned to the same 10° blocks), so the tests stay honest. The ablation "C without rain-fed counter-examples" (same learner and folds, counter-examples switched off) shows what they add. It is reported, not ranked as a candidate.

In [ ]:
def bal_w(y):
    p = float(np.mean(y))
    return np.where(y == 1, 0.5 / p, 0.5 / (1 - p))

class ConstantModel:
    """Fallback when a training fold holds only one class (happens for rare crops on irrigated-only subsets)."""
    def __init__(self, p):
        self.p = float(p)
    def predict_proba(self, X):
        return np.column_stack([np.full(len(X), 1 - self.p), np.full(len(X), self.p)])

def counter_rows(X, desert_mask=None):
    """Rain-fed counter-examples, all labelled 'not grown' with irrigation share 0 (the last column of X):
    (1) copies of the fully irrigated cells in X (share >= counter_irr_share): MapSPAM shows no rain-fed farming there;
    (2) unfarmed deserts (X_DESERT rows selected by desert_mask; None = all): almost nothing is farmed there at all."""
    assert X.shape[1] == X_DESERT.shape[1]
    Xc = X[X[:, -1] >= CFG['counter_irr_share']].copy()
    Xc[:, -1] = 0.0
    return np.vstack([Xc, X_DESERT if desert_mask is None else X_DESERT[desert_mask]])

def fit_model(make, X, y, weighted=True, X_counter=None):
    """Class-balanced fit: presences and absences carry equal total weight, so 0.5 means 'as typical of cells that
    grow the crop as of cells that do not'. Works for plain estimators and pipelines. X_counter: extra 'not grown' rows."""
    if X_counter is not None and len(X_counter):
        X, y = np.vstack([X, X_counter]), np.concatenate([y, np.zeros(len(X_counter))])
    if len(np.unique(y)) < 2:
        return ConstantModel(float(np.mean(y)) if len(y) else 0.0)
    m = make()
    if not weighted:
        return m.fit(X, y)
    last = m.steps[-1][0] if hasattr(m, 'steps') else None
    return m.fit(X, y, **({f'{last}__sample_weight': bal_w(y)} if last else {'sample_weight': bal_w(y)}))

def model_nodes(m):
    """Number of tree nodes in a fitted model (its size in the numpy runtime); NaN for non-tree models."""
    try:
        if hasattr(m, 'estimators_'):
            return float(sum(e.tree_.node_count for e in m.estimators_))
        if hasattr(m, '_predictors'):
            return float(sum(len(p[0].nodes) for p in m._predictors))
        if hasattr(m, 'get_booster'):
            return float(sum(d.count('\n') for d in m.get_booster().get_dump()))
        if hasattr(m, 'booster_'):
            return float(sum(2 * t['num_leaves'] - 1 for t in m.booster_.dump_model()['tree_info']))
    except Exception:
        pass
    return np.nan

def run_crop(make, X, y, has_irr=True, allowed=None, weighted=True, egypt=False, perm_groups=None, seed=SEED, counter=True, X_desert=None):
    """Spatial CV for one crop. Returns out-of-fold predictions at the cell's real irrigation share ('oof'), in irrigated
    mode ('oof1', share = 1) and rain-fed mode ('oof0', share = 0). Optionally: predictions on Egypt from a model trained
    without Egypt ('egypt', irrigated mode) and grouped permutation importance (drop in held-out AUC).
    counter: add rain-fed counter-examples from the training fold (only for designs with the irrigation input).
    X_desert: unfarmed-desert rows in the same columns as X (default X_DESERT for designs with the irrigation input);
    each fold's model predicts its held-out deserts ('oofD', rain-fed mode)."""
    n = len(y)
    o, o1, o0 = (np.full(n, np.nan) for _ in range(3))
    if X_desert is None and has_irr and X.shape[1] == X_DESERT.shape[1]:
        X_desert = X_DESERT
    oD = np.full(len(DESERT_FOLD), np.nan)
    lab = ~np.isnan(y) & (True if allowed is None else allowed)
    use_counter = has_irr and counter
    rng = np.random.default_rng(seed)
    drops = {g: [] for g in (perm_groups or {})}
    sizes = []
    for k_fold, (tr, te) in enumerate(FOLDS):
        m = fit_model(make, X[tr[lab[tr]]], y[tr[lab[tr]]], weighted, counter_rows(X[tr], DESERT_FOLD != k_fold) if use_counter else None)
        sizes.append(model_nodes(m))
        o[te] = m.predict_proba(X[te])[:, 1]
        if X_desert is not None and (DESERT_FOLD == k_fold).any():
            oD[DESERT_FOLD == k_fold] = m.predict_proba(X_desert[DESERT_FOLD == k_fold])[:, 1]
        if has_irr:
            X1, X0 = X[te].copy(), X[te].copy()
            X1[:, -1], X0[:, -1] = 1.0, 0.0
            o1[te], o0[te] = m.predict_proba(X1)[:, 1], m.predict_proba(X0)[:, 1]
        else:
            o1[te] = o0[te] = o[te]
        if perm_groups:
            k = te[~np.isnan(y[te])]
            if 0 < y[k].sum() < len(k):
                base = roc_auc_score(y[k], o[k])
                for g, cols in perm_groups.items():
                    Xp = X[k].copy()
                    perm = rng.permutation(len(k))
                    Xp[:, cols] = Xp[perm][:, cols]        # permute the whole group jointly (keeps within-group links)
                    drops[g].append(base - roc_auc_score(y[k], m.predict_proba(Xp)[:, 1]))
    res = {'oof': o, 'oof1': o1, 'oof0': o0, 'oofD': oD, 'nodes': float(np.mean(sizes))}
    if egypt:
        tr = np.where(lab & ~EGYPT)[0]
        m = fit_model(make, X[tr], y[tr], weighted, counter_rows(X[~EGYPT], ~DESERT_EGYPT) if use_counter else None)
        Xe = X[EGYPT].copy()
        if has_irr:
            Xe[:, -1] = 1.0
        res['egypt'] = m.predict_proba(Xe)[:, 1]
    if perm_groups:
        res['perm'] = {g: float(np.mean(v)) if v else np.nan for g, v in drops.items()}
    return res

def run_all_crops(make, X, labels, **kw):
    """run_crop for every crop in parallel (one single-threaded model per worker)."""
    out = joblib.Parallel(n_jobs=N_JOBS)(joblib.delayed(run_crop)(make, X, labels[c].to_numpy(), **kw) for c in CROPS)
    return dict(zip(CROPS, out))

def crop_metrics(y, r):
    """Held-out metrics for one crop from run_crop output."""
    ok = ~np.isnan(y)
    yy, pp = y[ok], r['oof'][ok]
    pos, neg = yy == 1, yy == 0
    out = {'AUC': roc_auc_score(yy, pp),
           'TSS': float((pp[pos] >= CFG['suit_threshold']).mean() - (pp[neg] >= CFG['suit_threshold']).mean()),
           'Brier (balanced)': float(np.average((pp - yy) ** 2, weights=bal_w(yy)))}
    for key, mask, p in [('AUC irrigated farmland', IRR_DOM, r['oof1']), ('AUC rain-fed farmland', RF_DOM, r['oof0'])]:
        k = ok & mask
        out[key] = roc_auc_score(y[k], p[k]) if (y[k] == 1).sum() >= 20 and (y[k] == 0).sum() >= 20 else np.nan
    out['Rain-fed false alarms (dry irrigated cells)'] = float((r['oof0'][DRY_IRR] >= CFG['suit_threshold']).mean())
    out['Rain-fed false alarms (unfarmed deserts)'] = float((r['oofD'] >= CFG['suit_threshold']).mean()) if not np.isnan(r['oofD']).all() else np.nan
    return out

FAO_ITEMS = {
    'wheat': ['Wheat'], 'rice': ['Rice'], 'maize': ['Maize (corn)'], 'barley': ['Barley'], 'pearl millet': ['Millet'],
    'small millets': ['Millet'], 'sorghum': ['Sorghum'], 'potato': ['Potatoes'], 'sweet potato': ['Sweet potatoes'],
    'yams': ['Yams'], 'cassava': ['Cassava, fresh'], 'beans': ['Beans, dry'], 'chickpea': ['Chick peas, dry'],
    'cowpea': ['Cow peas, dry'], 'pigeon pea': ['Pigeon peas, dry'], 'lentil': ['Lentils, dry'], 'soybean': ['Soya beans'],
    'groundnut': ['Groundnuts, excluding shelled'], 'coconut': ['Coconuts, in shell'], 'oil palm': ['Oil palm fruit'],
    'sunflower': ['Sunflower seed'], 'rapeseed': ['Rape or colza seed'], 'sesame': ['Sesame seed'], 'sugarcane': ['Sugar cane'],
    'sugar beet': ['Sugar beet'], 'cotton': ['Seed cotton, unginned'], 'arabica coffee': ['Coffee, green'],
    'robusta coffee': ['Coffee, green'], 'cocoa': ['Cocoa beans'], 'tea': ['Tea leaves'], 'tobacco': ['Unmanufactured tobacco'],
    'banana': ['Bananas'], 'plantain': ['Plantains and cooking bananas'],
    'citrus': ['Oranges', 'Tangerines, mandarins, clementines', 'Lemons and limes', 'Pomelos and grapefruits', 'Other citrus fruit, n.e.c.'],
    'tropical fruit': ['Mangoes, guavas and mangosteens', 'Papayas', 'Avocados', 'Pineapples', 'Other tropical fruits, n.e.c.'],
    'temperate fruit': ['Apples', 'Pears', 'Peaches and nectarines', 'Apricots', 'Plums and sloes', 'Cherries', 'Quinces'],
    'tomato': ['Tomatoes'], 'onion': ['Onions and shallots, dry (excluding dehydrated)'],
    'vegetables': ['Other vegetables, fresh n.e.c.', 'Eggplants (aubergines)', 'Chillies and peppers, green (Capsicum spp. and Pimenta spp.)',
                   'Cucumbers and gherkins', 'Pumpkins, squash and gourds', 'Cabbages', 'Carrots and turnips', 'Okra',
                   'Cauliflowers and broccoli', 'Lettuce and chicory', 'Spinach', 'Artichokes'],
    'rubber': ['Natural rubber in primary forms']}
# Fallback: FAOSTAT Egypt mean area harvested 2015-2024 (ha), summed over the items above; used only if the file is not attached.
FAO_FALLBACK_HA = {'wheat': 1376460, 'maize': 1024209, 'rice': 537008, 'vegetables': 328223, 'sugar beet': 237049, 'potato': 213914,
                   'citrus': 207547, 'tomato': 170616, 'sorghum': 158088, 'sugarcane': 144638, 'tropical fruit': 140844, 'cotton': 99543,
                   'onion': 88878, 'groundnut': 65684, 'temperate fruit': 61982, 'beans': 41185, 'sesame': 35422, 'banana': 30148,
                   'barley': 25070, 'soybean': 22788, 'sunflower': 14580, 'sweet potato': 12960, 'cowpea': 1982, 'chickpea': 1303, 'lentil': 497}
if FAO_CSV is not None:
    fao = pd.read_csv(FAO_CSV)
    last = int(fao['Year'].max())
    fao_area = fao[(fao['Element'] == 'Area harvested') & (fao['Year'] > last - 10)].groupby('Item')['Value'].mean()
    fao_ha = {c: float(sum(fao_area.get(i, 0.0) for i in FAO_ITEMS[c])) for c in CROPS}
    FAO_PERIOD = f'{last - 9} to {last}'
else:
    fao_ha = {c: float(FAO_FALLBACK_HA.get(c, 0.0)) for c in CROPS}
    FAO_PERIOD = '2015 to 2024 (embedded)'
FAO_TRUTH = pd.DataFrame({'egypt_area_ha': fao_ha})
FAO_TRUTH['grown_in_egypt'] = np.where(FAO_TRUTH.egypt_area_ha >= CFG['fao_grown_min_ha'], 1.0,
                                       np.where(FAO_TRUTH.egypt_area_ha <= CFG['fao_absent_max_ha'], 0.0, np.nan))
print(f'FAOSTAT {FAO_PERIOD}: grown at scale {int((FAO_TRUTH.grown_in_egypt == 1).sum())} crops | not grown '
      f'{int((FAO_TRUTH.grown_in_egypt == 0).sum())} | in between (left out) {int(FAO_TRUTH.grown_in_egypt.isna().sum())}')
W_EG = farm.loc[EGYPT, 'TA_total'].to_numpy()

def mean_auc(y_tab, P, mask=None, min_n=20):
    out = []
    for c in CROPS:
        y, p = y_tab[c].to_numpy(), P[c]
        k = ~np.isnan(y) & ~np.isnan(p) & (True if mask is None else mask)
        if (y[k] == 1).sum() >= min_n and (y[k] == 0).sum() >= min_n:
            out.append(roc_auc_score(y[k], p[k]))
    return float(np.mean(out)) if out else np.nan

def egypt_tests(eg_pred):
    """eg_pred: {crop: predictions on Egypt farmland cells, irrigated mode} from a model that never saw Egypt."""
    yE = LAB.loc[EGYPT]
    yy = np.concatenate([yE[c].to_numpy() for c in CROPS]); pp = np.concatenate([eg_pred[c] for c in CROPS])
    k = ~np.isnan(yy)
    nat = pd.Series({c: float(np.average(eg_pred[c], weights=W_EG)) for c in CROPS})
    lab = FAO_TRUTH.join(nat.rename('score')).dropna(subset=['grown_in_egypt'])
    g = lab.grown_in_egypt == 1
    hit, fa = float((lab.score[g] >= CFG['suit_threshold']).mean()), float((lab.score[~g] >= CFG['suit_threshold']).mean())
    return {'Egypt cells: AUC crop x cell (MapSPAM)': roc_auc_score(yy[k], pp[k]), 'FAOSTAT AUC (grown vs not)': roc_auc_score(lab.grown_in_egypt, lab.score),
            'FAOSTAT grown rated suitable': hit, 'FAOSTAT not-grown rated suitable': fa, 'FAOSTAT TSS': hit - fa}, nat

def lgbm_fixed():
    return LGBMClassifier(n_estimators=150, learning_rate=0.1, num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                          n_jobs=1, random_state=SEED, verbose=-1)

THERMAL = ['t_mean', 't_coldest', 't_warmest', 'tmax_hottest']
LAB_RF = share_labels(farm, prefix='TR_')
DESIGNS = {   # name: (features, labels, irrigation input?, rows allowed in training, role, rain-fed counter-examples?)
    'R: rain-fed crops only, all 10 features': (FEATURES, LAB_RF, False, None, 'rainfed', False),
    'TH: irrigated farmland only, temperature + humidity': (THERMAL + ['rh_mean'], LAB, False, (farm.irr_share >= 0.5).to_numpy(), 'irrigated', False),
    'A: all crops, all 10 features, no irrigation input': (FEATURES, LAB, False, None, 'both', False),
    'C without rain-fed counter-examples': (COLS, LAB, True, None, 'both', False),
    'C: all crops, all 10 features + irrigation share': (COLS, LAB, True, None, 'both', True),
    'C with hectare labels (>= 1,000 ha present)': (COLS, hectare_labels(farm), True, None, 'both', True),
}
FA_COL = 'Rain-fed false alarms (dry irrigated cells)'
FA_DESERT_COL = 'Rain-fed false alarms (unfarmed deserts)'
t0 = time.time()
design_rows, design_nat = {}, {}
for name, (feats, L, has_irr, allowed, role, counter) in DESIGNS.items():
    Xd = farm[feats].to_numpy(float)
    Xdes = desert[feats].to_numpy(float)                    # irr_share is 0 for every desert row
    R = run_all_crops(lgbm_fixed, Xd, L, has_irr=has_irr, allowed=allowed, egypt=(role != 'rainfed'), counter=counter, X_desert=Xdes)
    oof1, oof0 = {c: R[c]['oof1'] for c in CROPS}, {c: R[c]['oof0'] for c in CROPS}
    r = {'CV AUC irrigated farmland': mean_auc(LAB, oof1, IRR_DOM) if role != 'rainfed' else np.nan,
         'CV AUC rain-fed farmland': mean_auc(LAB, oof0, RF_DOM) if role != 'irrigated' else np.nan,
         FA_COL: float(np.mean([(oof0[c][DRY_IRR] >= CFG['suit_threshold']).mean() for c in CROPS])) if role != 'irrigated' else np.nan,
         FA_DESERT_COL: float(np.mean([(R[c]['oofD'] >= CFG['suit_threshold']).mean() for c in CROPS])) if role != 'irrigated' else np.nan}
    if role != 'rainfed':
        e, nat = egypt_tests({c: R[c]['egypt'] for c in CROPS})
        r.update(e)
        design_nat[name] = nat
    design_rows[name] = r
    print(f'{name}: done ({time.time() - t0:.0f} s)')
DESIGN = pd.DataFrame(design_rows).T
display(DESIGN.round(3))
ABLATION = 'C without rain-fed counter-examples'
DESIGN_CAND = DESIGN.drop(index=[k for k, v in DESIGNS.items() if v[4] == 'rainfed'] + [ABLATION])
score_cols = ['CV AUC irrigated farmland', 'Egypt cells: AUC crop x cell (MapSPAM)', 'FAOSTAT AUC (grown vs not)', 'FAOSTAT TSS']
_r3 = DESIGN_CAND[score_cols + [FA_COL, FA_DESERT_COL]].astype(float).round(3)    # differences in the 4th decimal count as ties
DESIGN_RANK = pd.concat([_r3[score_cols].rank(ascending=False), _r3[[FA_COL, FA_DESERT_COL]].rank(ascending=True)], axis=1).mean(axis=1).sort_values()
SELECTED_DESIGN = DESIGN_RANK.index[0]
print('Mean rank over the four irrigated-mode tests and the two rain-fed desert tests (1 = best):', DESIGN_RANK.round(2).to_dict())
print('Selected design:', SELECTED_DESIGN)
# ablation: what do the counter-examples change? (same learner and folds, counter-examples switched off)
_with, _without = DESIGN.loc['C: all crops, all 10 features + irrigation share'], DESIGN.loc[ABLATION]
COUNTER_ABLATION = pd.DataFrame({'with counter-examples': _with, 'without': _without, 'change': _with - _without}).loc[
    ['CV AUC irrigated farmland', 'CV AUC rain-fed farmland', FA_COL, FA_DESERT_COL, 'Egypt cells: AUC crop x cell (MapSPAM)', 'FAOSTAT AUC (grown vs not)', 'FAOSTAT TSS']]
display(COUNTER_ABLATION.astype(float).round(3))
_auc_cost = float(-COUNTER_ABLATION.loc[['CV AUC irrigated farmland', 'CV AUC rain-fed farmland'], 'change'].min())
print(f"Counter-examples: rain-fed false alarms on dry irrigated cells {_without[FA_COL]:.3f} -> {_with[FA_COL]:.3f}, "
      f"on unfarmed deserts {_without[FA_DESERT_COL]:.3f} -> {_with[FA_DESERT_COL]:.3f}; largest AUC loss {max(_auc_cost, 0):.3f} "
      f"-> {'kept' if _with[FA_COL] < _without[FA_COL] and _with[FA_DESERT_COL] < _without[FA_DESERT_COL] and _auc_cost <= 0.01 else 'CHECK: they cost skill here'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7.2), gridspec_kw={'width_ratios': [1.25, 1]})
ax = axes[0]
DPLOT = DESIGN.assign(**{'rain-fed: dry irrigated cells rated unsuitable': 1 - DESIGN[FA_COL],
                         'rain-fed: unfarmed deserts rated unsuitable': 1 - DESIGN[FA_DESERT_COL]})
cols_plot = ['CV AUC irrigated farmland', 'CV AUC rain-fed farmland', 'rain-fed: dry irrigated cells rated unsuitable',
             'rain-fed: unfarmed deserts rated unsuitable', 'Egypt cells: AUC crop x cell (MapSPAM)', 'FAOSTAT AUC (grown vs not)', 'FAOSTAT TSS']
short = ['CV AUC\nirrigated\nfarmland', 'CV AUC\nrain-fed\nfarmland', 'rain-fed mode:\ndry irrigated\ncells unsuitable',
         'rain-fed mode:\nunfarmed deserts\nunsuitable', 'Egypt cells\nAUC (left out)', 'FAOSTAT\nEgypt AUC', 'FAOSTAT\nEgypt TSS']
w = 0.13
colors = ['#9e9ac8', '#fdae6b', '#a1d99b', '#fcbba1', '#08519c', '#bdbdbd']
for k, (name, row) in enumerate(DPLOT.iterrows()):
    ax.bar(np.arange(len(cols_plot)) + (k - (len(DPLOT) - 1) / 2) * w, row[cols_plot].to_numpy(float), w, color=colors[k], label=name,
           edgecolor='k' if name == SELECTED_DESIGN else 'none', lw=1.2)
ax.set_xticks(np.arange(len(cols_plot))); ax.set_xticklabels(short, fontsize=7.5)
ax.set_ylim(0, 1); ax.set_ylabel('score (higher is better)')
ax.set_title(f'A. {len(DESIGN_CAND) + 1} designs + 1 ablation, one learner (LightGBM), held-out tests only')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=True)
ax = axes[1]
nat_sel = design_nat[SELECTED_DESIGN]
tt = FAO_TRUTH.join(nat_sel.rename('score')).dropna(subset=['grown_in_egypt']).sort_values('score')
ax.barh(tt.index, tt.score, color=np.where(tt.grown_in_egypt == 1, '#238b45', '#bdbdbd'))
ax.axvline(CFG['suit_threshold'], c='k', ls='--', lw=1)
ax.set_xlabel('irrigated suitability, mean over Egypt farmland (model never saw Egypt)')
ax.set_title('B. Selected design on Egypt: green = grown at scale (FAOSTAT)')
ax.tick_params(axis='y', labelsize=7)
fig.suptitle('Figure 3. Choosing what to learn: irrigation as a model input wins', fontweight='bold', y=1.02)
fig.tight_layout()
savefig(fig, 'fig03_design_selection')

**Decision (design).** Design **C** learns one model per crop from all 10 NASA features **plus the irrigation share** of the farmland, with rain-fed counter-examples. It has the best mean rank over the held-out tests (the printed ranking above). It also needs only one model for both app modes: irrigated mode sets the input to 1, rain-fed mode to 0. Four findings shaped it:
1. **Share labels beat hectare labels.** With a fixed 1,000 ha rule, the model rates almost nothing suitable in Egypt, because Egypt's farmland is a narrow strip and most cells hold little farmland.
2. **Irrigation must be an input.** Without it (design A), dry-climate cells mix irrigated and rain-fed farms. With it, the model separates "too dry for rain-fed" from "fine when watered".
3. **Counter-examples make rain-fed mode honest in deserts.** Without them, rain-fed mode calls many crops suitable on dry, fully irrigated farmland and in empty deserts the model never saw. With them, those false alarms drop sharply while AUC barely moves (ablation table above).
4. **Temperature + humidity alone (design TH)** is the classic way to model irrigated farming. It is slightly worse than C on most held-out tests and has no rain-fed mode.

**What the model still misses on held-out Egypt (Figure 3B).** Water-hungry summer crops (rice, sugarcane, cotton and some oilseeds) score low when Egypt is left out. That is not a temperature problem. Outside Egypt, fully irrigated desert farmland is mostly small groundwater farms that grow vegetables, fruit and wheat. Hardly any desert has a river like the Nile. The deployed model is trained on all cells, Egypt included, so it learns this from Egypt's own farmland. The leave-Egypt-out numbers remain our honest estimate for a region the model has never seen.

## 6. Choosing the algorithm: eight classical machine-learning families
With the design fixed (one model per crop, 10 NASA features + irrigation share), we compare eight families of classical ML. They get the same data, the same class-balanced weights and the same 5-fold spatial cross-validation (40 crops x 5 folds = 200 fits per family). Every number below comes from regions the model did not see during training.

| Metric | Question it answers |
|---|---|
| **AUC** | Does the model rank cells that grow the crop above cells that do not? 0.5 = random, 1 = perfect. |
| **AUC irrigated / rain-fed farmland** | The same, in each app mode: irrigated farmland scored with share = 1, rain-fed farmland with share = 0. |
| **TSS** (true skill statistic) | Sensitivity + specificity - 1 at the app's threshold of 0.5 (Allouche et al. 2006). 0 = no skill, 1 = perfect. |
| **Brier score (class-balanced)** | Mean squared error of the predicted value. A proper scoring rule that punishes over-confidence (lower is better). |
| **Rain-fed false alarms (dry irrigated cells)** | Share of crops rated suitable in rain-fed mode on held-out farmland that is ≥ 95 % irrigated and gets < 100 mm of rain a year. MapSPAM shows no rain-fed farming there (lower is better). |
| **Rain-fed false alarms (unfarmed deserts)** | The same count on held-out desert cells with < 100 mm of rain and < 1,000 ha farmed (lower is better). |

We also record each model's **size**: the number of tree nodes the app would have to ship for all 40 crops. The app runs the model with numpy only. Its budget is 2 million nodes (`max_app_nodes`), about 20 MB.

In [ ]:
from functools import partial

class BalancedKNN:
    """k-nearest neighbours with the same class balance as the weighted models (kNN cannot take sample weights):
    the neighbour vote is re-weighted by the inverse class frequency of the training cells."""
    def __init__(self, n_neighbors=50, weights='distance'):
        self.n_neighbors, self.weights = n_neighbors, weights
    def fit(self, X, y, sample_weight=None):
        self.pi_ = float(np.mean(y))
        self.m_ = make_pipeline(StandardScaler(), KNeighborsClassifier(self.n_neighbors, weights=self.weights)).fit(X, y)
        return self
    def predict_proba(self, X):
        p = self.m_.predict_proba(X)[:, 1]
        a, b = p / self.pi_, (1 - p) / (1 - self.pi_)
        q = a / np.maximum(a + b, 1e-12)
        return np.column_stack([1 - q, q])

# name: (builder(**params), default params used in the zoo, random-search space used for tuning)
FAMILIES = {
    'Logistic regression (quadratic)': (
        lambda **p: make_pipeline(StandardScaler(), PolynomialFeatures(2), StandardScaler(), LogisticRegression(max_iter=300, tol=1e-3, **p)),
        {'C': 0.1}, {'C': [0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]}),
    'Gaussian naive Bayes': (lambda **p: GaussianNB(**p), {}, {'var_smoothing': [1e-9, 1e-7, 1e-5, 1e-3, 1e-2, 1e-1]}),
    'k-nearest neighbours': (lambda **p: BalancedKNN(**p), {'n_neighbors': 50, 'weights': 'distance'},
                             {'n_neighbors': [10, 25, 50, 100, 200], 'weights': ['uniform', 'distance']}),
    'Random forest': (lambda **p: RandomForestClassifier(n_jobs=1, random_state=SEED, **p),
                      {'n_estimators': 100, 'min_samples_leaf': 5, 'max_features': 'sqrt'},
                      {'n_estimators': [40, 60, 100, 150], 'min_samples_leaf': [5, 10, 20, 50], 'max_features': ['sqrt', 0.5, 0.8],
                       'max_depth': [8, 10, 12, 14, 16, None]}),
    'Extra trees': (lambda **p: ExtraTreesClassifier(n_jobs=1, random_state=SEED, **p),
                    {'n_estimators': 100, 'min_samples_leaf': 5, 'max_features': 'sqrt'},
                    {'n_estimators': [40, 60, 100, 150], 'min_samples_leaf': [5, 10, 20, 50], 'max_features': ['sqrt', 0.5, 0.8],
                     'max_depth': [8, 10, 12, 14, 16, None]}),
    'HistGradientBoosting': (lambda **p: HistGradientBoostingClassifier(random_state=SEED, **p),
                             {'max_iter': 300, 'learning_rate': 0.05},
                             {'max_iter': [200, 300, 500], 'learning_rate': [0.03, 0.05, 0.1], 'max_leaf_nodes': [7, 15, 31, 63],
                              'min_samples_leaf': [20, 50, 100, 200], 'l2_regularization': [0.0, 0.1, 1.0, 10.0]}),
}
if HAS_XGB:
    FAMILIES['XGBoost'] = (lambda **p: XGBClassifier(tree_method='hist', n_jobs=1, random_state=SEED, verbosity=0, **p),
                           {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.8},
                           {'n_estimators': [200, 300, 500], 'learning_rate': [0.02, 0.03, 0.05, 0.1], 'max_depth': [3, 4, 5, 6, 8],
                            'subsample': [0.6, 0.8, 1.0], 'colsample_bytree': [0.6, 0.8, 1.0], 'min_child_weight': [1, 5, 20, 50],
                            'reg_lambda': [1, 5, 20]})
if HAS_LGBM:
    FAMILIES['LightGBM'] = (lambda **p: LGBMClassifier(subsample_freq=1, n_jobs=1, random_state=SEED, verbose=-1, **p),
                            {'n_estimators': 300, 'learning_rate': 0.05, 'num_leaves': 31, 'subsample': 0.8, 'colsample_bytree': 0.8},
                            {'n_estimators': [200, 300, 500], 'learning_rate': [0.02, 0.03, 0.05, 0.1], 'num_leaves': [7, 15, 31, 63],
                             'min_child_samples': [20, 50, 100, 200], 'subsample': [0.6, 0.8, 1.0], 'colsample_bytree': [0.6, 0.8, 1.0],
                             'reg_lambda': [0, 1, 5, 20]})
METRIC_COLS = ['AUC', 'AUC irrigated farmland', 'AUC rain-fed farmland', 'TSS', 'Brier (balanced)', FA_COL, FA_DESERT_COL]
if QUICK:
    FAMILIES = {k: v for k, v in FAMILIES.items() if k in ('Gaussian naive Bayes', 'Extra trees', 'XGBoost', 'LightGBM')}
HIGHER_BETTER = {'AUC': True, 'AUC irrigated farmland': True, 'AUC rain-fed farmland': True, 'TSS': True, 'Brier (balanced)': False,
                 FA_COL: False, FA_DESERT_COL: False}

def evaluate(make, crops=None, **kw):
    """Spatial CV for the given crops -> (per-crop metrics table incl. model size in nodes, raw run outputs)."""
    crops = crops or CROPS
    runs = joblib.Parallel(n_jobs=N_JOBS)(joblib.delayed(run_crop)(make, X_ALL, LAB[c].to_numpy(), **kw) for c in crops)
    per = pd.DataFrame([crop_metrics(LAB[c].to_numpy(), r) for c, r in zip(crops, runs)], index=crops)
    per['nodes'] = [r['nodes'] for r in runs]
    return per, dict(zip(crops, runs))

def mean_rank(tab):
    return pd.concat([tab[c].rank(ascending=not HIGHER_BETTER[c]) for c in METRIC_COLS], axis=1).mean(axis=1)

ZOO_PER, ZOO_TIME = {}, {}
for name, (builder, params, _) in FAMILIES.items():
    t0 = time.time()
    ZOO_PER[name], _ = evaluate(partial(builder, **params))
    ZOO_TIME[name] = time.time() - t0
    s = ZOO_PER[name].mean()
    print(f'{name:32s} AUC {s["AUC"]:.4f} | irrigated {s["AUC irrigated farmland"]:.4f} | rain-fed {s["AUC rain-fed farmland"]:.4f} | '
          f'TSS {s["TSS"]:.3f} | Brier {s["Brier (balanced)"]:.4f} | rain-fed false alarms {s[FA_COL]:.3f} / deserts {s[FA_DESERT_COL]:.3f} | {ZOO_TIME[name]:.0f} s')
ZOO_TAB = pd.DataFrame({n: p.mean() for n, p in ZOO_PER.items()}).T[METRIC_COLS]
ZOO_TAB['worst-crop AUC'] = [ZOO_PER[n]['AUC'].min() for n in ZOO_TAB.index]
ZOO_TAB['app size (M nodes, 40 crops)'] = [ZOO_PER[n]['nodes'].sum() / 1e6 for n in ZOO_TAB.index]
ZOO_TAB['seconds'] = pd.Series(ZOO_TIME)
ZOO_TAB['mean rank'] = mean_rank(ZOO_TAB)
ZOO_TAB = ZOO_TAB.sort_values('mean rank')
TOP3 = ZOO_TAB.index[:3].tolist()
display(ZOO_TAB.round(4))
print(f'Top 3 families (mean rank over the {len(METRIC_COLS)} metrics):', TOP3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), gridspec_kw={'width_ratios': [1.2, 1.2, 0.9]})
order = ZOO_TAB.index.tolist()[::-1]
ax = axes[0]
for col, mk, cc in [('AUC', 'o', '#08519c'), ('AUC irrigated farmland', 's', '#9e9ac8'), ('AUC rain-fed farmland', '^', '#fdae6b'), ('TSS', 'D', '#31a354')]:
    ax.plot(ZOO_TAB.loc[order, col], range(len(order)), mk, c=cc, ms=7, label=col)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
ax.set_xlabel('score (higher is better)'); ax.set_xlim(0.3, 1.0)
ax.set_title('A. Mean over 40 crops, spatial cross-validation')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=True)
ax = axes[1]
ax.boxplot([ZOO_PER[n]['AUC'].to_numpy() for n in order], vert=False, widths=.6, showfliers=True,
           flierprops=dict(ms=3), medianprops=dict(color='#08519c'))
ax.set_yticks(range(1, len(order) + 1)); ax.set_yticklabels(order)
ax.set_xlabel('held-out AUC per crop'); ax.set_title('B. Spread over crops (each box = 40 crops)')
ax = axes[2]
ax.barh(range(len(order)), ZOO_TAB.loc[order, 'Brier (balanced)'], color=['#08519c' if n in TOP3 else '#bdbdbd' for n in order])
for i, n in enumerate(order):
    sz = ZOO_TAB.loc[n, 'app size (M nodes, 40 crops)']
    ax.text(ZOO_TAB.loc[n, 'Brier (balanced)'] + .003, i, f"{ZOO_TAB.loc[n, 'seconds']:.0f} s" + (f" | {sz:.1f} M nodes" if np.isfinite(sz) else ''),
            va='center', fontsize=7)
ax.set_yticks(range(len(order))); ax.set_yticklabels([])
ax.set_xlim(0, ZOO_TAB['Brier (balanced)'].max() * 1.6)
ax.set_xlabel('class-balanced Brier score (lower is better)'); ax.set_title('C. Calibration error (label: run time | app size)')
fig.suptitle('Figure 4. Model zoo: eight classical ML families on the same spatial cross-validation (blue = top 3, tuned next)',
             fontweight='bold', y=1.02)
fig.tight_layout()
savefig(fig, 'fig04_model_zoo')

## 7. Tuning the three best families
Each of the top three gets a random search over its main hyper-parameters (`ParameterSampler`, `tune_iter` candidates plus its zoo default). To keep the run time reasonable, candidates are scored on 12 crops spread over the range of prevalence, from the most widely grown to the rarest, with the same spatial folds. The objective is the **class-balanced Brier score**. It rewards ranking and honest probabilities at the same time, which matters because the app compares crops with each other and uses a fixed 0.5 threshold.

**Constraint:** a candidate can only be selected if its 40-crop model fits the app's size budget (`max_app_nodes`). Deep forests can need over 10 million nodes. The forest search therefore includes compact settings (fewer and shallower trees, larger leaves).

In [ ]:
n_present = (LAB == 1).sum().sort_values()
TUNE_CROPS = n_present.index[np.unique(np.linspace(0, len(n_present) - 1, CFG['tune_crops']).round().astype(int))].tolist()
print('Tuning crops:', ', '.join(f'{c} ({n_present[c]})' for c in TUNE_CROPS))

def score_candidate(builder, params, crop):
    r = run_crop(partial(builder, **params), X_ALL, LAB[crop].to_numpy())
    return {**crop_metrics(LAB[crop].to_numpy(), r), 'nodes': r['nodes']}

TUNING, BEST_PARAMS = {}, {}
for name in TOP3:
    builder, default, space = FAMILIES[name]
    cands = [dict(default)] + [{**default, **p} for p in ParameterSampler(space, n_iter=CFG['tune_iter'], random_state=SEED)]
    t0 = time.time()
    res = joblib.Parallel(n_jobs=N_JOBS)(joblib.delayed(score_candidate)(builder, p, c) for p in cands for c in TUNE_CROPS)
    rows = []
    for k, p in enumerate(cands):
        m = pd.DataFrame(res[k * len(TUNE_CROPS):(k + 1) * len(TUNE_CROPS)]).mean()
        app_nodes = m['nodes'] * len(CROPS)
        rows.append({'candidate': 'zoo default' if k == 0 else f'random {k}', 'params': p, **m.drop('nodes').to_dict(),
                     'app size (M nodes)': app_nodes / 1e6, 'fits budget': bool(app_nodes <= CFG['max_app_nodes'])})
    tab = pd.DataFrame(rows).sort_values(['fits budget', 'Brier (balanced)'], ascending=[False, True]).reset_index(drop=True)
    if not tab.loc[0, 'fits budget']:
        print(f'  {name}: no candidate fits the size budget; its smallest candidate is used in the final comparison.')
        tab = tab.sort_values('app size (M nodes)').reset_index(drop=True)
    TUNING[name] = tab
    BEST_PARAMS[name] = tab.loc[0, 'params']
    print(f"{name}: {len(cands)} candidates in {time.time() - t0:.0f} s | best Brier {tab.loc[0, 'Brier (balanced)']:.4f} "
          f"(default {tab[tab.candidate == 'zoo default']['Brier (balanced)'].iloc[0]:.4f}) | size {tab.loc[0, 'app size (M nodes)']:.2f} M nodes | "
          f"best params {BEST_PARAMS[name]}")
display(pd.concat({n: t[['candidate', 'Brier (balanced)', 'AUC', 'TSS', 'app size (M nodes)', 'fits budget']].head(5)
                   for n, t in TUNING.items()}).round(4))

In [ ]:
fig, axes = plt.subplots(1, len(TOP3), figsize=(5.6 * len(TOP3), 4.4), sharey=True)
for ax, name in zip(np.atleast_1d(axes), TOP3):
    tab = TUNING[name]
    ok_ = tab['fits budget']
    ax.scatter(tab['AUC'][ok_], tab['Brier (balanced)'][ok_], s=40, c='#9ecae1', edgecolors='#08519c', label='candidate within size budget')
    ax.scatter(tab['AUC'][~ok_], tab['Brier (balanced)'][~ok_], s=40, marker='x', c='#cb181d', label='too large for the app')
    d = tab[tab.candidate == 'zoo default']
    ax.scatter(d['AUC'], d['Brier (balanced)'], s=90, marker='s', facecolors='none', edgecolors='k', label='zoo default')
    ax.scatter(tab['AUC'].iloc[:1], tab['Brier (balanced)'].iloc[:1], s=220, marker='*', c='#e6550d', label='selected (lowest Brier in budget)')
    ax.set_title(name); ax.set_xlabel(f'mean held-out AUC ({len(TUNE_CROPS)} tuning crops)')
np.atleast_1d(axes)[0].set_ylabel('class-balanced Brier score (lower is better)')
np.atleast_1d(axes)[0].legend(frameon=True, loc='upper right')
fig.suptitle('Figure 5. Random-search tuning of the top three families (spatial cross-validation)', fontweight='bold', y=1.02)
fig.tight_layout()
savefig(fig, 'fig05_tuning')

## 8. Final comparison on all 40 crops and choice of the algorithm
The three tuned models are compared on all 40 crops. The same run also produces two extra results: the **leave-Egypt-out** predictions (model retrained without any Egyptian cell) and the **grouped permutation importance** of each NASA measurement. The algorithm is chosen on the global cross-validation metrics only. Egypt stays a test, never a training target.

In [ ]:
PERM_GROUPS = {'T2M (temperature)': [COLS.index(f) for f in ['t_mean', 't_coldest', 't_warmest']],
               'T2M_MAX (heat)': [COLS.index('tmax_hottest')],
               'PRECTOTCORR (rain)': [COLS.index(f) for f in ['rain_annual', 'rain_wettest3_share']],
               'RH2M (humidity)': [COLS.index('rh_mean')],
               'GWETTOP (surface soil wetness)': [COLS.index('soil_top_wetness')],
               'GWETROOT (root-zone soil wetness)': [COLS.index(f) for f in ['soil_root_wetness', 'soil_root_wetness_min']],
               'irrigation share (MapSPAM)': [COLS.index('irr_share')]}
FINAL_PER, FINAL_RUNS, FINAL_EGYPT, FINAL_NAT = {}, {}, {}, {}
for name in TOP3:
    t0 = time.time()
    per, runs = evaluate(partial(FAMILIES[name][0], **BEST_PARAMS[name]), egypt=True, perm_groups=PERM_GROUPS)
    FINAL_PER[name], FINAL_RUNS[name] = per, runs
    FINAL_EGYPT[name], FINAL_NAT[name] = egypt_tests({c: runs[c]['egypt'] for c in CROPS})
    print(f'{name} (tuned): {time.time() - t0:.0f} s')
FINAL_TAB = pd.DataFrame({n: FINAL_PER[n].mean() for n in TOP3}).T[METRIC_COLS]
FINAL_TAB['worst-crop AUC'] = [FINAL_PER[n]['AUC'].min() for n in TOP3]
FINAL_TAB['app size (M nodes)'] = [FINAL_PER[n]['nodes'].sum() / 1e6 for n in TOP3]
FINAL_TAB['mean rank (CV)'] = mean_rank(FINAL_TAB)
EGYPT_TAB = pd.DataFrame(FINAL_EGYPT).T
FINAL_TAB = FINAL_TAB.sort_values(['mean rank (CV)', 'Brier (balanced)'])
PACKABLE = {'LightGBM', 'XGBoost', 'HistGradientBoosting', 'Random forest', 'Extra trees'}   # families the numpy runtime can evaluate
deployable = [n for n in FINAL_TAB.index if n in PACKABLE]
FINAL_NAME = deployable[0]
if FINAL_NAME != FINAL_TAB.index[0]:
    print(f'Note: {FINAL_TAB.index[0]} ranks first but has no numpy-only runtime; using the best tree ensemble.')
FINAL_PARAMS = BEST_PARAMS[FINAL_NAME]
make_final = partial(FAMILIES[FINAL_NAME][0], **FINAL_PARAMS)
display(FINAL_TAB.round(4))
display(EGYPT_TAB.loc[FINAL_TAB.index].round(3))
print(f'Selected algorithm: {FINAL_NAME} with {FINAL_PARAMS}')
# the reason, from the numbers above: best model per metric and the mean rank that decides
_dec_cols = METRIC_COLS + ['worst-crop AUC']
_hb = {**HIGHER_BETTER, 'worst-crop AUC': True}
_best = {c: (FINAL_TAB[c].idxmax() if _hb[c] else FINAL_TAB[c].idxmin()) for c in _dec_cols}
ALGO_DECISION = {'selected': FINAL_NAME, 'mean_rank': FINAL_TAB['mean rank (CV)'].round(2).to_dict(),
                 'best_per_metric': {c: {'model': _best[c], 'value': float(FINAL_TAB.loc[_best[c], c])} for c in _dec_cols},
                 'metrics_won_by_selected': [c for c in _dec_cols if _best[c] == FINAL_NAME],
                 'largest_gap_to_selected': {c: float((FINAL_TAB[c] - FINAL_TAB.loc[FINAL_NAME, c]).abs().max()) for c in _dec_cols}}
print('Best tuned model per metric (spatial CV):')
for c in _dec_cols:
    print(f"  {c:48s} {_best[c]:22s} {FINAL_TAB.loc[_best[c], c]:.4f}   (selected: {FINAL_TAB.loc[FINAL_NAME, c]:.4f})")
_others = [n for n in FINAL_TAB.index if n != FINAL_NAME]
print(f"Decision: {FINAL_NAME} " + ('has the best mean rank' if FINAL_NAME == FINAL_TAB.index[0] else 'is the best-ranked model the numpy runtime can run') +
      f" over the {len(METRIC_COLS)} metrics "
      f"({FINAL_TAB.loc[FINAL_NAME, 'mean rank (CV)']:.2f} vs " + ', '.join(f"{n} {FINAL_TAB.loc[n, 'mean rank (CV)']:.2f}" for n in _others) + "). "
      f"It is best on {len(ALGO_DECISION['metrics_won_by_selected'])} of {len(_dec_cols)} metrics; "
      f"the largest AUC gap between the three is {ALGO_DECISION['largest_gap_to_selected']['AUC']:.3f}. Calibration (ECE) follows in section 9.")

**Decision (algorithm).** The rule is fixed before the run: the tuned model with the best **mean rank** over the seven cross-validation metrics wins, ties are broken by the Brier score, and only tree ensembles the app's numpy runtime can evaluate are eligible. The printed lines above show which model is best on each metric and by how much. The three tuned ensembles are close, so the choice matters less than the design choice in section 5. The runners-up stay in the comparison tables and in `metrics.json`.

In [ ]:
per = FINAL_PER[FINAL_NAME].join(n_present.rename('present cells')).sort_values('AUC')
fig, axes = plt.subplots(1, 2, figsize=(17, 8.2), sharey=True)
ax = axes[0]
yy_ = np.arange(len(per))
ax.barh(yy_, per['AUC'], color='#c6dbef', label='AUC, all farmland (real irrigation share)')
ax.plot(per['AUC irrigated farmland'], yy_, 's', c='#6a51a3', ms=4, label='AUC irrigated farmland (share = 1)')
ax.plot(per['AUC rain-fed farmland'], yy_, '^', c='#e6550d', ms=4, label='AUC rain-fed farmland (share = 0)')
ax.set_yticks(yy_); ax.set_yticklabels([f'{c} ({int(n):,})' for c, n in zip(per.index, per['present cells'])], fontsize=7)
ax.set_xlim(0.5, 1.0); ax.set_xlabel('held-out AUC (spatial cross-validation)')
ax.set_title(f'A. Ranking skill per crop (label: cells where it is grown)')
ax.legend(loc='lower right', frameon=True)
ax = axes[1]
ax.barh(yy_, per['TSS'], color='#a1d99b')
ax.axvline(per['TSS'].mean(), c='k', ls='--', lw=1)
ax.text(per['TSS'].mean() + .01, len(per) - 1, f"mean {per['TSS'].mean():.2f}", fontsize=8, va='top')
ax.set_xlim(0, 1); ax.set_xlabel('TSS at the app threshold 0.5 (sensitivity + specificity - 1)')
ax.set_title('B. Yes/no skill at the threshold the app uses')
fig.suptitle(f'Figure 6. Selected model ({FINAL_NAME}, tuned): held-out skill for each of the 40 crops', fontweight='bold', y=1.0)
fig.tight_layout()
savefig(fig, 'fig06_per_crop_skill')

In [ ]:
def reliability(runs, key='oof', bins=np.linspace(0, 1, 11)):
    """Pooled over crops, with each crop's class-balanced weights: in a well-calibrated balanced model,
    cells predicted p contain presences with balanced frequency p."""
    P, Y, W = [], [], []
    for c in CROPS:
        y = LAB[c].to_numpy(); k = ~np.isnan(y)
        P.append(runs[c][key][k]); Y.append(y[k]); W.append(bal_w(y[k]) / k.sum())
    P, Y, W = map(np.concatenate, (P, Y, W))
    b = np.clip(np.digitize(P, bins) - 1, 0, len(bins) - 2)
    out = pd.DataFrame({'p': P, 'y': Y, 'w': W, 'bin': b}).groupby('bin').apply(
        lambda g: pd.Series({'mean_pred': np.average(g.p, weights=g.w), 'observed': np.average(g.y, weights=g.w), 'weight': g.w.sum()}))
    ece = float((out.weight * (out.mean_pred - out.observed).abs()).sum() / out.weight.sum())
    return out, ece, P
CALIB = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for name, cc in zip(TOP3, ['#08519c', '#e6550d', '#31a354']):
    rel, ece, P = reliability(FINAL_RUNS[name])
    CALIB[name] = ece
    axes[0].plot(rel.mean_pred, rel.observed, 'o-', c=cc, label=f'{name} (ECE {ece:.3f})', lw=2 if name == FINAL_NAME else 1)
    axes[1].hist(P, bins=40, histtype='step', color=cc, lw=1.5, density=True, label=name)
axes[0].plot([0, 1], [0, 1], 'k:', lw=1)
axes[0].set_xlabel('predicted suitability (held-out)'); axes[0].set_ylabel('observed presence rate (class-balanced)')
axes[0].set_title('A. Reliability: points on the diagonal = honest values'); axes[0].legend(frameon=True)
axes[1].set_xlabel('predicted suitability'); axes[1].set_ylabel('density'); axes[1].set_title('B. How the predictions are spread')
fig.suptitle('Figure 7. Calibration of the tuned models (spatial cross-validation, 40 crops pooled)', fontweight='bold', y=1.02)
fig.tight_layout()
savefig(fig, 'fig07_calibration')
print('Expected calibration error:', {k: round(v, 4) for k, v in CALIB.items()})

## 9. What the model learned
**Which NASA measurements matter?** For each crop and each held-out fold, all features from one NASA parameter are shuffled together. The resulting drop in AUC is that parameter's importance. Shuffling them together keeps derived features, such as mean and coldest-month temperature, from masking each other.

**How to read it.** The score measures what a measurement adds *that the others cannot replace*. Soil wetness is closely tied to rain and humidity. When it is shuffled, the model can still read most of the same information from rain and humidity, so its score stays small even though it carries real signal. Temperature has no such stand-in.

In [ ]:
IMP = pd.DataFrame({c: FINAL_RUNS[FINAL_NAME][c]['perm'] for c in CROPS}).T
imp_mean = IMP.mean().sort_values()
fig, axes = plt.subplots(1, 2, figsize=(18, 7.8), gridspec_kw={'width_ratios': [1, 1.5]})
ax = axes[0]
ax.barh(imp_mean.index, imp_mean, color='#9ecae1', edgecolor='#08519c')
for i, g in enumerate(imp_mean.index):
    ax.plot(IMP[g], np.full(len(IMP), i) + np.random.default_rng(i).uniform(-.25, .25, len(IMP)), '.', c='#08519c', ms=3, alpha=.6)
ax.set_xlabel('drop in held-out AUC when the measurement is shuffled (bar = mean, dots = crops)')
ax.set_title('A. Importance of each NASA measurement')
ax = axes[1]
H_ = IMP[imp_mean.index[::-1]].loc[IMP.mean(axis=1).sort_values(ascending=False).index]
im = ax.imshow(H_.T.to_numpy(), aspect='auto', cmap='Blues', vmin=0)
ax.set_yticks(range(H_.shape[1])); ax.set_yticklabels(H_.columns, fontsize=8)
ax.set_xticks(range(H_.shape[0])); ax.set_xticklabels(H_.index, rotation=90, fontsize=7)
ax.grid(False)
plt.colorbar(im, ax=ax, shrink=.7, label='drop in AUC')
ax.set_title('B. Per crop')
fig.suptitle(f'Figure 8. Which NASA measurements the model relies on ({FINAL_NAME}, grouped permutation importance)', fontweight='bold', y=1.0)
fig.tight_layout()
savefig(fig, 'fig08_importance_by_nasa_parameter')
display(imp_mean[::-1].round(4).to_frame('mean drop in AUC'))

**Does the map look right?** Below: where four contrasting crops are grown according to MapSPAM, next to the model's held-out suitability (each region predicted by a model that never saw it).

In [ ]:
MAP_CROPS = [c for c in ['wheat', 'rice', 'cotton', 'cassava'] if c in CROPS]
fig, axes = plt.subplots(len(MAP_CROPS), 2, figsize=(17, 3.6 * len(MAP_CROPS)))
cat_cmap = ListedColormap(['#d9d9d9', '#fdd0a2', '#238b45'])   # not grown, uncertain, grown
for row, c in zip(np.atleast_2d(axes), MAP_CROPS):
    y = LAB[c].to_numpy()
    ax = row[0]
    ax.imshow(grid_image(farm, np.where(np.isnan(y), 1, np.where(y == 1, 2, 0))), extent=GRID_EXTENT, origin='lower',
              cmap=cat_cmap, vmin=-.5, vmax=2.5, interpolation='nearest')
    ax.set_title(f'{c}: MapSPAM 2020 (green = grown, grey = not grown, orange = uncertain)')
    cb = plt.colorbar(plt.cm.ScalarMappable(cmap=cat_cmap, norm=plt.Normalize(-.5, 2.5)), ax=ax, shrink=.8, ticks=[0, 1, 2])
    cb.ax.set_yticklabels(['not grown', 'uncertain', 'grown'])
    ax = row[1]
    m = ax.imshow(grid_image(farm, FINAL_RUNS[FINAL_NAME][c]['oof']), extent=GRID_EXTENT, origin='lower', cmap='YlGn', vmin=0, vmax=1,
                  interpolation='nearest')
    ax.set_title(f'{c}: model suitability, held-out (AUC {FINAL_PER[FINAL_NAME].loc[c, "AUC"]:.2f})')
    plt.colorbar(m, ax=ax, shrink=.8, label='suitability')
    for ax in row:
        ax.set_xlim(-180, 180); ax.set_ylim(-58, 72); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.suptitle('Figure 9. Learned from the map, tested on unseen regions: MapSPAM (left) vs model (right)', fontweight='bold', y=1.0)
fig.tight_layout()
savefig(fig, 'fig09_world_maps_heldout')

## 10. The deployable model and its numpy-only runtime
The app model is retrained per crop on **all** labelled farmland cells, Egypt included, with the tuned settings. The trees are then packed into flat numpy arrays (`models/crop_suitability_v2.npz`). `crop_engine_v2.py` evaluates them with numpy alone, so the API needs neither scikit-learn nor LightGBM/XGBoost. The engine file is written by this notebook (below). The notebook then loads it and checks that it reproduces the library predictions.

In [ ]:
ENGINE_SRC = r'''"""crop_engine_v2.py - CropShift Earth crop-suitability runtime (v2: learns from all six NASA POWER measurements).

Generated by the Kaggle notebook "CropShift Earth v2". Retrain + regenerate, do not hand-edit.
Dependencies: numpy (+ Python standard library for live NASA POWER requests). No scikit-learn / LightGBM needed at runtime.

    import crop_engine_v2 as ce
    eng = ce.CropSuitabilityEngine.load("crop_suitability_v2.npz")
    report = ce.point_report(eng, lat=30.04, lon=31.24)      # live NASA POWER (6 parameters), any point on Earth
    report = ce.point_report(eng, 30.04, 31.24, irrigation_level=0.5)   # irrigation slider (irrigated mode)
    eng.rank(features_dict, mode="irrigated")                 # features: see ce.FEATURES
    reports = ce.batch_reports(eng, [(key, lat, lon, parameter), ...])  # a whole region, one model call per mode

The model was trained on where 40 crops are actually grown (IFPRI MapSPAM 2020, about 22,000 farmland cells worldwide),
described by 10 climate features computed from the six NASA POWER measurements (T2M, T2M_MAX, PRECTOTCORR, RH2M, GWETTOP, GWETROOT).
"""
from __future__ import annotations

import json
import math
import urllib.parse
import urllib.request

import numpy as np

P_STRONG, P_MODERATE = 0.01, 0.05
SUIT_THRESHOLD = 0.5
CLASS_EDGES = (0.33, 0.67)
POWER_URL = "https://power.larc.nasa.gov/api/temporal/monthly/point"
POWER_PARAMS = ["T2M", "T2M_MAX", "PRECTOTCORR", "RH2M", "GWETTOP", "GWETROOT"]
PROJECTION_LABEL = "Projection if the observed trend continues (Theil-Sen extrapolation). Not a forecast."

FEATURES = ['t_mean', 't_coldest', 't_warmest', 'tmax_hottest', 'rain_annual', 'rain_wettest3_share',
            'rh_mean', 'soil_top_wetness', 'soil_root_wetness', 'soil_root_wetness_min']
FEATURE_INFO = {
    't_mean': ('T2M', 'annual mean temperature', 'deg C'),
    't_coldest': ('T2M', 'mean temperature of the coldest month', 'deg C'),
    't_warmest': ('T2M', 'mean temperature of the warmest month', 'deg C'),
    'tmax_hottest': ('T2M_MAX', 'highest monthly maximum temperature of the year (heat stress)', 'deg C'),
    'rain_annual': ('PRECTOTCORR', 'annual rainfall', 'mm/year'),
    'rain_wettest3_share': ('PRECTOTCORR', 'share of the annual rain falling in the wettest 3 consecutive months', '0-1'),
    'rh_mean': ('RH2M', 'annual mean relative humidity', '%'),
    'soil_top_wetness': ('GWETTOP', 'annual mean surface soil wetness', '0-1'),
    'soil_root_wetness': ('GWETROOT', 'annual mean root-zone soil wetness', '0-1'),
    'soil_root_wetness_min': ('GWETROOT', 'root-zone soil wetness of the driest month', '0-1'),
}
FEATURE_BOUNDS = {'rain_annual': (0.0, None), 'rain_wettest3_share': (0.25, 1.0), 'rh_mean': (0.0, 100.0),
                  'soil_top_wetness': (0.0, 1.0), 'soil_root_wetness': (0.0, 1.0), 'soil_root_wetness_min': (0.0, 1.0)}
_DAYS = np.array([31, 28.25, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])


def annual_features(T, TX, P, RH, GT, GR):
    """Each argument: array (..., 12) of monthly NASA POWER values for one year
    (T2M, T2M_MAX in deg C; PRECTOTCORR in mm/day; RH2M in %; GWETTOP, GWETROOT 0-1). Returns (..., 10) in FEATURES order."""
    T, TX, P, RH, GT, GR = (np.asarray(a, dtype=np.float64) for a in (T, TX, P, RH, GT, GR))
    rain_m = P * _DAYS
    rain_y = rain_m.sum(-1)
    ext = np.concatenate([rain_m, rain_m[..., :2]], axis=-1)
    q3 = ext[..., 0:12] + ext[..., 1:13] + ext[..., 2:14]
    share = np.where(rain_y > 1.0, q3.max(-1) / np.maximum(rain_y, 1e-9), 1.0)
    return np.stack([T.mean(-1), T.min(-1), T.max(-1), TX.max(-1), rain_y, share,
                     RH.mean(-1), GT.mean(-1), GR.mean(-1), GR.min(-1)], axis=-1)


def clip_features(F):
    F = np.array(F, dtype=np.float64, copy=True)
    for k, (lo, hi) in FEATURE_BOUNDS.items():
        j = FEATURES.index(k)
        F[..., j] = np.clip(F[..., j], lo if lo is not None else -np.inf, hi if hi is not None else np.inf)
    return F


def suitability_class(s):
    if s is None or not np.isfinite(s):
        return None
    return "high" if s >= CLASS_EDGES[1] else ("medium" if s >= CLASS_EDGES[0] else "low")


def _r(x, nd=2):
    x = float(x)
    return None if not math.isfinite(x) else round(x, nd)


# ---------------------------------------------------------------- packed tree ensembles (numpy evaluator)
class _Pack:
    """All crops' trees in flat arrays; trees are grouped by crop (tree_crop is sorted)."""

    def __init__(self, z, prefix):
        g = lambda k: z[f"{prefix}{k}"]
        self.feature, self.threshold = g("feature").astype(np.intp), g("threshold")
        self.left, self.right, self.value = g("left"), g("right"), g("value")
        self.default_left, self.is_leaf = g("default_left"), g("is_leaf")
        self.roots, self.tree_crop = g("roots"), g("tree_crop")
        self.kind = str(g("kind"))
        self.base = g("base")                     # per-crop base score
        # XGBoost and scikit-learn forests compare float32 inputs: round the same way to reproduce them exactly
        self.f32 = bool(z[f"{prefix}input_f32"]) if f"{prefix}input_f32" in z.files else False
        self.n_crops = int(self.base.size)
        self.start = np.searchsorted(self.tree_crop, np.arange(self.n_crops))
        self.count = np.bincount(self.tree_crop, minlength=self.n_crops)

    def predict(self, X, chunk=128):
        X = np.atleast_2d(np.asarray(X, np.float64))
        if self.f32:
            X = X.astype(np.float32).astype(np.float64)
        out = np.empty((len(X), self.n_crops))
        for s in range(0, len(X), chunk):
            x = X[s:s + chunk]
            node = np.broadcast_to(self.roots[:, None], (self.roots.size, len(x))).copy()
            cols = np.arange(len(x))[None, :]
            while True:
                act = ~self.is_leaf[node]
                if not act.any():
                    break
                xv = x[cols, self.feature[node]]
                go_left = np.where(np.isnan(xv), self.default_left[node], xv <= self.threshold[node])
                node = np.where(act, np.where(go_left, self.left[node], self.right[node]), node)
            sums = np.add.reduceat(self.value[node], self.start, axis=0).T          # (n, n_crops)
            if self.kind == "gbdt":
                out[s:s + chunk] = 1.0 / (1.0 + np.exp(-(sums + self.base[None, :])))
            else:
                out[s:s + chunk] = sums / self.count[None, :]
        return out


class CropSuitabilityEngine:
    """s[crop](10 NASA climate features, irrigation). Modes: 'irrigated' (water supplied) and 'rainfed'."""

    def __init__(self, z):
        self.meta = json.loads(str(z["meta_json"]))
        self.version = self.meta["version"]
        self.crops = list(self.meta["crops"])
        self.n_crops = len(self.crops)
        self.modes = {}
        packs = {}
        for mode, spec in self.meta["modes"].items():
            if spec["prefix"] not in packs:
                packs[spec["prefix"]] = _Pack(z, spec["prefix"])
            self.modes[mode] = {"pack": packs[spec["prefix"]], "features": [FEATURES.index(f) for f in spec["features"]],
                                "irrigation_input": spec.get("irrigation_input"), "irrigation_value": spec.get("irrigation_value")}
        self.train_lo = np.array(self.meta["training_range"]["min"], float)
        self.train_hi = np.array(self.meta["training_range"]["max"], float)
        self._native = {}

    @classmethod
    def load(cls, path):
        return cls(np.load(path, allow_pickle=False))

    def attach_native(self, prefix, fn):
        """Optional speed-up for bulk runs: fn(X) -> (n, n_crops) from the original library models.
        The numpy evaluator stays the reference; the notebook checks both agree before attaching."""
        self._native[prefix] = fn

    def suitability(self, F, mode="irrigated", irrigation_level=1.0):
        """F: (n, 10) features in FEATURES order -> (n, n_crops) suitability in [0, 1]."""
        if mode not in self.modes:
            raise ValueError(f"mode must be one of {list(self.modes)}")
        F = clip_features(np.atleast_2d(np.asarray(F, np.float64)))
        spec = self.modes[mode]
        X = F[:, spec["features"]]
        if spec["irrigation_input"] is not None:
            fixed = spec.get("irrigation_value")
            lvl = float(fixed) if fixed is not None else float(np.clip(irrigation_level, 0.0, 1.0))
            X = np.column_stack([X, np.full(len(X), lvl)])
        bad = ~np.isfinite(X).all(axis=1)
        Xc = np.where(np.isfinite(X), X, 0.0)
        fn = self._native.get(self.meta["modes"][mode]["prefix"])
        out = np.asarray(fn(Xc), float) if fn is not None else spec["pack"].predict(Xc)
        out[bad] = np.nan
        return np.clip(out, 0.0, 1.0)

    def in_training_range(self, F):
        """True where every feature lies inside the range seen in the training farmland (min to max)."""
        F = np.atleast_2d(np.asarray(F, float))
        return ((F >= self.train_lo) & (F <= self.train_hi)).all(axis=1)

    def rank(self, features, mode="irrigated", irrigation_level=1.0):
        F = np.array([[features[k] for k in FEATURES]], float) if isinstance(features, dict) else np.atleast_2d(features)
        s = self.suitability(F, mode, irrigation_level)[0]
        return [{"crop": self.crops[k], "suitability": _r(s[k], 3), "class": suitability_class(s[k])} for k in np.argsort(-s)]


# ---------------------------------------------------------------- trend statistics (same as v1)
def theil_sen_batch(Y, t, chunk=20000):
    Y = np.atleast_2d(np.asarray(Y, float))
    t = np.asarray(t, float)
    i, j = np.triu_indices(t.size, 1)
    dt_ = t[j] - t[i]
    b = np.empty(len(Y))
    for s in range(0, len(Y), chunk):
        b[s:s + chunk] = np.median((Y[s:s + chunk, j] - Y[s:s + chunk, i]) / dt_, axis=1)
    a = np.median(Y - b[:, None] * t[None, :], axis=1)
    return a, b


def mann_kendall_batch(Y, decimals=None, chunk=20000):
    Y = np.atleast_2d(np.asarray(Y, float))
    if decimals is not None:
        Y = np.round(Y, decimals)
    n = Y.shape[1]
    i, j = np.triu_indices(n, 1)
    S = np.empty(len(Y))
    for s in range(0, len(Y), chunk):
        S[s:s + chunk] = np.sign(Y[s:s + chunk, j] - Y[s:s + chunk, i]).sum(axis=1)
    var = np.full(len(Y), n * (n - 1) * (2 * n + 5) / 18.0)
    for k in range(len(Y)):
        _, cnt = np.unique(Y[k], return_counts=True)
        cnt = cnt[cnt > 1]
        if cnt.size:
            var[k] -= float((cnt * (cnt - 1) * (2 * cnt + 5)).sum()) / 18.0
    with np.errstate(divide="ignore", invalid="ignore"):
        z = np.where(var > 0, (S - np.sign(S)) / np.sqrt(np.where(var > 0, var, 1.0)), 0.0)
    p = np.array([math.erfc(abs(v) / math.sqrt(2.0)) for v in z])
    return S, z, p


def confidence_labels(p, slope=None, min_effect=0.0):
    p = np.atleast_1d(np.asarray(p, float))
    lab = np.where(p < P_STRONG, "strong evidence", np.where(p < P_MODERATE, "moderate evidence", "not proven"))
    lab = np.where(np.isfinite(p), lab, "not proven")
    if slope is not None and min_effect > 0:
        lab = np.where(np.abs(np.atleast_1d(slope)) < min_effect, "not proven", lab)
    return lab


def direction_labels(slope, conf, up="gaining", down="losing", none="no proven change"):
    slope = np.atleast_1d(np.asarray(slope, float))
    return np.where(np.asarray(conf) == "not proven", none, np.where(slope > 0, up, down))


def block_bootstrap_indices(n, n_boot=200, block_len=3, seed=42):
    rng = np.random.default_rng(seed)
    n_blocks = math.ceil(n / block_len)
    starts = rng.integers(0, n - block_len + 1, size=(n_boot, n_blocks))
    return (starts[:, :, None] + np.arange(block_len)[None, None, :]).reshape(n_boot, -1)[:, :n]


def project_series(Y, years, base_year, horizons, idx):
    """Theil-Sen trend extrapolation with residual block bootstrap. The same year indices are used for every
    row, so the features keep their joint year-to-year behaviour (a hot year stays a dry year)."""
    Y = np.atleast_2d(np.asarray(Y, float))
    t = np.asarray(years, float) - float(base_year)
    a, b = theil_sen_batch(Y, t)
    fit = a[:, None] + b[:, None] * t[None, :]
    res = Y - fit
    ystar = fit[:, None, :] + res[:, idx]
    a_s, b_s = theil_sen_batch(ystar.reshape(-1, t.size), t)
    H = np.asarray(horizons, float)
    point = a[:, None] + b[:, None] * H[None, :]
    draws = a_s.reshape(len(Y), -1)[..., None] + b_s.reshape(len(Y), -1)[..., None] * H
    return {"point": point, "draws": draws, "slope": b}


# ---------------------------------------------------------------- NASA POWER
def fetch_power_point(lat, lon, start=2001, end=2025, timeout=90):
    q = urllib.parse.urlencode({"parameters": ",".join(POWER_PARAMS), "community": "AG", "longitude": f"{lon:.4f}",
                                "latitude": f"{lat:.4f}", "start": int(start), "end": int(end), "format": "JSON"})
    with urllib.request.urlopen(f"{POWER_URL}?{q}", timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))["properties"]["parameter"]


def yearly_features(parameter):
    """parameter: {'T2M': {'YYYYMM': v}, ...} for the six POWER_PARAMS -> years (n,), features (n, 10).
    Only complete years (12 valid months of all six measurements) are used."""
    keys = sorted(k for k in parameter["T2M"] if k[4:] != "13")
    years = sorted({int(k[:4]) for k in keys})
    ys, rows = [], []
    for y in years:
        M = np.full((6, 12), np.nan)
        for m in range(1, 13):
            for vi, v in enumerate(POWER_PARAMS):
                val = parameter.get(v, {}).get(f"{y}{m:02d}")
                if val is not None and np.isfinite(val) and val > -900:
                    M[vi, m - 1] = float(val)
        if np.isfinite(M).all():
            ys.append(y)
            rows.append(annual_features(*M))
    return np.array(ys, int), np.array(rows, float).reshape(-1, len(FEATURES))


# ---------------------------------------------------------------- report (API payload, v1-compatible + features block)
def _prepare(parameter, base_year=None, horizons=(5, 10), n_boot=200, block_len=3, seed=42):
    """Yearly features, Mann-Kendall per feature and Theil-Sen projections with block-bootstrap draws."""
    years, Fy = yearly_features(parameter)
    if len(years) < 10:
        raise ValueError("need at least 10 complete years of all six NASA POWER measurements (land points only)")
    base_year = int(years[-1]) if base_year is None else int(base_year)
    horizons = [int(h) for h in horizons]
    H = np.array([0] + horizons, float)
    idx = block_bootstrap_indices(len(years), n_boot, block_len, seed)
    proj = project_series(Fy.T, years, base_year, H, idx)
    pt = clip_features(proj["point"].T)                              # (H, 10)
    dr = clip_features(np.transpose(proj["draws"], (1, 2, 0)))       # (B, H, 10)
    _, _, p_f = mann_kendall_batch(Fy.T)
    return {"years": years, "Fy": Fy, "base_year": base_year, "horizons": horizons, "H": H, "n_boot": int(n_boot),
            "proj": proj, "pt": pt, "dr": dr, "p_f": p_f}


def _rows(prep):
    """All feature rows one report needs: trend values (H), bootstrap draws (B x H), observed years."""
    return np.vstack([prep["pt"], prep["dr"].reshape(-1, len(FEATURES)), prep["Fy"]])


def _split(prep, S):
    nH, nB = len(prep["H"]), prep["n_boot"]
    return S[:nH], S[nH:nH + nB * nH].reshape(nB, nH, -1), S[nH + nB * nH:]


def _in_polygon(lat, lon, poly):
    """Ray-casting point-in-polygon test; poly is a list of (lon, lat) vertices."""
    inside, n = False, len(poly)
    for i in range(n):
        (x1, y1), (x2, y2) = poly[i], poly[(i + 1) % n]
        if (y1 > lat) != (y2 > lat) and lon < x1 + (lat - y1) * (x2 - x1) / (y2 - y1):
            inside = not inside
    return inside


def data_flags_for(engine, lat, lon):
    """Data-quality flags stored in the model file (e.g. a possible data break in one NASA measurement over a region)
    that apply at this point."""
    return [f for f in engine.meta.get("data_flags", []) if not f.get("polygon") or _in_polygon(lat, lon, f["polygon"])]


def _assemble(engine, lat, lon, prep, scores, irrigation_level=1.0, include_series=True,
              min_effect_suit=0.02, min_effect_options=0.10, top=8):
    years, Fy, pt, dr, proj, p_f = (prep[k] for k in ("years", "Fy", "pt", "dr", "proj", "p_f"))
    base_year, horizons, H, n_boot = prep["base_year"], prep["horizons"], prep["H"], prep["n_boot"]
    conf_f = confidence_labels(p_f)
    feats = {}
    for k, name in enumerate(FEATURES):
        src, desc, unit = FEATURE_INFO[name]
        e = {"nasa_parameter": src, "description": desc, "unit": unit, "now": _r(pt[0, k], 3),
             "trend_per_decade": _r(proj["slope"][k] * 10, 4), "p_value": _r(p_f[k], 4), "confidence": str(conf_f[k])}
        for hi, h in enumerate(horizons, start=1):
            d = dr[:, hi, k]
            e[f"plus{h}"] = {"point": _r(pt[hi, k], 3), "median": _r(np.median(d), 3), "p10": _r(np.percentile(d, 10), 3),
                             "p90": _r(np.percentile(d, 90), 3)}
        if include_series:
            e["series"] = [_r(v, 3) for v in Fy[:, k]]
        feats[name] = e
    flags = data_flags_for(engine, lat, lon)
    for fl in flags:
        for name in fl.get("features", []):
            if name in feats:
                feats[name]["data_quality_note"] = fl["note"]
    # v1-compatible climate block (temperature in deg C, rainfall in mm/month)
    climate = {}
    for name, k, scale, unit in [("temperature_c", 0, 1.0, "deg C"), ("rainfall_mm_month", 4, 1 / 12.0, "mm/month")]:
        f = feats[FEATURES[k]]
        e = {"unit": unit, "now": _r(pt[0, k] * scale), "trend_per_decade": _r(proj["slope"][k] * 10 * scale, 4),
             "p_value": f["p_value"], "confidence": f["confidence"]}
        for hi, h in enumerate(horizons, start=1):
            d = dr[:, hi, k] * scale
            e[f"plus{h}"] = {"point": _r(pt[hi, k] * scale), "median": _r(np.median(d)), "p10": _r(np.percentile(d, 10)),
                             "p90": _r(np.percentile(d, 90))}
        if include_series:
            e["series"] = {"years": [int(y) for y in years], "values": [_r(v * scale, 3) for v in Fy[:, k]]}
        if "data_quality_note" in f:
            e["data_quality_note"] = f["data_quality_note"]
        climate[name] = e
    in_range = engine.in_training_range(pt)
    tyears = years - base_year
    modes = {}
    for mode, (s_pt, s_dr, s_ann) in scores.items():
        y2 = np.round(s_ann.T, 2)
        _, _, p_s = mann_kendall_batch(y2)
        _, b_s = theil_sen_batch(y2, tyears)
        conf_s = confidence_labels(p_s, b_s * 10, min_effect_suit)
        dir_s = direction_labels(b_s * 10, conf_s)
        crops = {}
        for ci, c in enumerate(engine.crops):
            e = {"now": _r(s_pt[0, ci], 3), "class": suitability_class(s_pt[0, ci]),
                 "trend": {"per_decade": _r(b_s[ci] * 10, 4), "p_value": _r(p_s[ci], 4),
                           "confidence": str(conf_s[ci]), "direction": str(dir_s[ci])}}
            now_in = bool(s_pt[0, ci] >= SUIT_THRESHOLD)
            for hi, h in enumerate(horizons, start=1):
                d = s_dr[:, hi, ci]
                p_cross = float(np.mean(d < SUIT_THRESHOLD)) if now_in else float(np.mean(d >= SUIT_THRESHOLD))
                e[f"plus{h}"] = {"point": _r(s_pt[hi, ci], 3), "median": _r(np.median(d), 3),
                                 "p10": _r(np.percentile(d, 10), 3), "p90": _r(np.percentile(d, 90), 3),
                                 "class": suitability_class(s_pt[hi, ci]), "p_cross": _r(p_cross, 3),
                                 "cross": (("loses suitability" if now_in else "becomes suitable") if p_cross >= 0.5 else None)}
            if include_series:
                e["series"] = [_r(v, 3) for v in s_ann[:, ci]]
            crops[c] = e
        ranking = {}
        for hi, h in enumerate(H):
            key = "now" if h == 0 else f"plus{int(h)}"
            ranking[key] = [{"crop": engine.crops[j], "suitability": _r(s_pt[hi, j], 3)} for j in np.argsort(-s_pt[hi])[:top]]
        opt_pt, opt_dr, opt_ann = s_pt.sum(1), s_dr.sum(2), s_ann.sum(1)
        top3 = np.sort(s_pt, axis=1)[:, -3:].mean(1)
        yv = np.round(opt_ann[None, :], 2)
        _, _, p_v = mann_kendall_batch(yv)
        _, b_v = theil_sen_batch(yv, tyears)
        conf_v = confidence_labels(p_v, b_v * 10, min_effect_options)
        via = {"crop_options_now": _r(opt_pt[0], 2), "viability_index_now": _r(opt_pt[0] / engine.n_crops, 3),
               "top3_mean_now": _r(top3[0], 3), "n_suitable_now": int((s_pt[0] >= SUIT_THRESHOLD).sum()),
               "trend": {"per_decade": _r(b_v[0] * 10, 3), "p_value": _r(p_v[0], 4), "confidence": str(conf_v[0]),
                         "direction": str(direction_labels(b_v * 10, conf_v)[0])}}
        for hi, h in enumerate(horizons, start=1):
            delta = opt_dr[:, hi] - opt_dr[:, 0]
            via[f"plus{h}"] = {"crop_options_point": _r(opt_pt[hi]), "crop_options_median": _r(np.median(opt_dr[:, hi])),
                               "p10": _r(np.percentile(opt_dr[:, hi], 10)), "p90": _r(np.percentile(opt_dr[:, hi], 90)),
                               "delta_median": _r(np.median(delta)), "delta_p10": _r(np.percentile(delta, 10)),
                               "delta_p90": _r(np.percentile(delta, 90)), "p_decline": _r(np.mean(delta < 0), 3),
                               "top3_mean_point": _r(top3[hi], 3), "n_suitable_point": int((s_pt[hi] >= SUIT_THRESHOLD).sum())}
        if include_series:
            via["series"] = [_r(v, 3) for v in opt_ann]
        modes[mode] = {"crops": crops, "ranking": ranking, "viability": via}
    rep = {"engine_version": engine.version, "location": {"lat": float(lat), "lon": float(lon)},
           "period": {"start": int(years[0]), "end": int(years[-1]), "base_year": base_year, "horizons": horizons},
           "irrigation_level": float(irrigation_level), "projection_label": PROJECTION_LABEL,
           "climate": climate, "features": feats,
           "in_training_range": {("now" if h == 0 else f"plus{int(h)}"): bool(v) for h, v in zip(H, in_range)},
           "modes": modes, "data_quality_notes": [fl["note"] for fl in flags]}
    rep["summary"] = summary_text(rep)
    return rep


def point_report(engine, lat, lon, parameter=None, start=2001, end=2025, base_year=None, horizons=(5, 10),
                 n_boot=200, block_len=3, seed=42, irrigation_level=1.0, include_series=True,
                 min_effect_suit=0.02, min_effect_options=0.10, top=8):
    """Full report for one point. parameter: NASA POWER monthly dict {'T2M': {'YYYYMM': v}, ...}; fetched live if None."""
    if parameter is None:
        parameter = fetch_power_point(lat, lon, start, end)
    prep = _prepare(parameter, base_year, horizons, n_boot, block_len, seed)
    R = _rows(prep)
    scores = {mode: _split(prep, engine.suitability(R, mode, irrigation_level)) for mode in engine.modes}
    return _assemble(engine, lat, lon, prep, scores, irrigation_level, include_series, min_effect_suit, min_effect_options, top)


def batch_reports(engine, points, base_year=None, horizons=(5, 10), n_boot=200, block_len=3, seed=42,
                  irrigation_level=1.0, include_series=False, min_effect_suit=0.02, min_effect_options=0.10, top=8):
    """points: iterable of (key, lat, lon, parameter). Same statistics as point_report (identical output per point),
    but the model is called once per mode for all points, which is much faster for a whole region."""
    points = list(points)
    preps = [_prepare(p, base_year, horizons, n_boot, block_len, seed) for _, _, _, p in points]
    rows = [_rows(pr) for pr in preps]
    cuts = np.cumsum([0] + [len(r) for r in rows])
    allrows = np.vstack(rows)
    S_all = {mode: engine.suitability(allrows, mode, irrigation_level) for mode in engine.modes}
    out = {}
    for i, ((key, lat, lon, _), pr) in enumerate(zip(points, preps)):
        scores = {mode: _split(pr, S[cuts[i]:cuts[i + 1]]) for mode, S in S_all.items()}
        out[key] = _assemble(engine, lat, lon, pr, scores, irrigation_level, include_series, min_effect_suit, min_effect_options, top)
    return out


def summary_text(rep):
    """Plain-language sentence built only from computed numbers (PROJECT_PLAN F1 wording rules)."""
    t, r = rep["climate"]["temperature_c"], rep["climate"]["rainfall_mm_month"]
    f = rep["features"]
    y0, h, by = rep["period"]["start"], rep["period"]["horizons"][-1], rep["period"]["base_year"]
    verb = "rose" if (t["trend_per_decade"] or 0) > 0 else "fell"
    out = [f"Since {y0}, average temperature here {verb} by {abs(t['trend_per_decade'] or 0):.2f} °C per decade ({t['confidence']}).",
           f"Rainfall changed by {r['trend_per_decade'] or 0:+.2f} mm/month per decade ({r['confidence']}); "
           f"root-zone soil wetness by {f['soil_root_wetness']['trend_per_decade'] or 0:+.3f} per decade "
           f"({f['soil_root_wetness']['confidence']})."]
    flagged = [lab for lab, e in (("temperature", t), ("rainfall", r)) if e.get("data_quality_note")]
    if flagged:
        out.append(f"NASA POWER {' and '.join(flagged)} here may contain a data break, so treat that trend with caution.")
    irr = rep["modes"]["irrigated"]
    n = len(irr["crops"])
    best = [x["crop"] for x in irr["ranking"]["now"][:3] if (x["suitability"] or 0) >= SUIT_THRESHOLD]
    out.append(f"With irrigation, the best-fitting crops today are {', '.join(best)}." if best
               else f"With irrigation, none of the {n} crops fits this climate well today.")
    gains = [c for c, e in irr["crops"].items() if e[f"plus{h}"]["cross"] == "becomes suitable"]
    losses = [c for c, e in irr["crops"].items() if e[f"plus{h}"]["cross"] == "loses suitability"]
    if gains or losses:
        parts = []
        if gains:
            parts.append("likely becomes suitable: " + ", ".join(gains))
        if losses:
            parts.append("likely loses suitability: " + ", ".join(losses))
        out.append(f"If current trends continue, by {by + h} " + "; ".join(parts) + ".")
    else:
        out.append(f"If current trends continue, no crop is expected to cross the suitability threshold by {by + h}.")
    if "rainfed" in rep["modes"]:
        out.append(f"Rain-fed: {rep['modes']['rainfed']['viability']['n_suitable_now']} of {n} crops fit today.")
    if not rep["in_training_range"].get("now", True):
        out.append("Note: this climate lies partly outside the range of the training farmland, so treat the ranking with care.")
    return " ".join(out)
'''
ENGINE_PATH = OUT / 'crop_engine_v2.py'
ENGINE_PATH.write_text(ENGINE_SRC, encoding='utf-8')
sys.dont_write_bytecode = True                              # keep __pycache__ out of the output folder
sys.path.insert(0, str(OUT))
if 'crop_engine_v2' in sys.modules:
    del sys.modules['crop_engine_v2']
import crop_engine_v2 as ce
assert ce.FEATURES == FEATURES
print('runtime engine written to', ENGINE_PATH, f'({ENGINE_PATH.stat().st_size / 1e3:.0f} kB)')

In [ ]:
def _finish(feat, thr, left, right, val, dleft, roots, kind, base, f32=False):
    feat = np.array(feat, np.int16)
    left, right = np.array(left, np.int32), np.array(right, np.int32)
    leaf = feat < 0
    left[leaf] = np.where(leaf)[0]; right[leaf] = np.where(leaf)[0]          # leaves point to themselves
    return {'feature': np.where(leaf, 0, feat).astype(np.int16), 'threshold': np.array(thr, np.float64), 'left': left, 'right': right,
            'value': np.array(val, np.float64), 'default_left': np.array(dleft, bool), 'is_leaf': leaf,
            'roots': np.array(roots, np.int32), 'kind': kind, 'base': float(base), 'f32': f32}

def pack_lightgbm(model):
    d = model.booster_.dump_model()
    feat, thr, left, right, val, dleft, roots = [], [], [], [], [], [], []
    def walk(node):
        k = len(feat)
        feat.append(-1); thr.append(0.0); left.append(-1); right.append(-1); val.append(0.0); dleft.append(True)
        if 'split_feature' not in node:
            val[k] = float(node['leaf_value']); return k
        assert node.get('decision_type', '<=') == '<='
        feat[k], thr[k], dleft[k] = int(node['split_feature']), float(node['threshold']), bool(node.get('default_left', True))
        left[k] = walk(node['left_child']); right[k] = walk(node['right_child'])
        return k
    for tr in d['tree_info']:
        roots.append(walk(tr['tree_structure']))
    return _finish(feat, thr, left, right, val, dleft, roots, 'gbdt', 0.0)

def pack_xgboost(model):
    """XGBoost tests float32(x) < split; stored as x <= previous float32 below split, with float32 inputs (flag f32)."""
    booster = model.get_booster()
    feat, thr, left, right, val, dleft, roots = [], [], [], [], [], [], []
    names = booster.feature_names
    fidx = (lambda s: names.index(s)) if names else (lambda s: int(str(s).lstrip('f')))
    def walk(node):
        k = len(feat)
        feat.append(-1); thr.append(0.0); left.append(-1); right.append(-1); val.append(0.0); dleft.append(True)
        if 'leaf' in node:
            val[k] = float(node['leaf']); return k
        ch = {c['nodeid']: c for c in node['children']}
        feat[k] = fidx(node['split'])
        thr[k] = float(np.nextafter(np.float32(node['split_condition']), np.float32(-np.inf)))
        dleft[k] = node['missing'] == node['yes']
        left[k] = walk(ch[node['yes']]); right[k] = walk(ch[node['no']])
        return k
    for s in booster.get_dump(dump_format='json'):
        roots.append(walk(json.loads(s)))
    bs = float(json.loads(booster.save_config())['learner']['learner_model_param']['base_score'].strip('[]'))
    return _finish(feat, thr, left, right, val, dleft, roots, 'gbdt', math.log(bs / (1 - bs)), f32=True)

def pack_hgb(model):
    feat, thr, left, right, val, dleft, roots = [], [], [], [], [], [], []
    for it in model._predictors:
        nodes, off = it[0].nodes, len(feat)
        roots.append(off)
        for nd in nodes:
            leaf = bool(nd['is_leaf'])
            feat.append(-1 if leaf else int(nd['feature_idx'])); thr.append(float(nd['num_threshold']))
            left.append(-1 if leaf else off + int(nd['left'])); right.append(-1 if leaf else off + int(nd['right']))
            val.append(float(nd['value']) if leaf else 0.0); dleft.append(bool(nd['missing_go_to_left']))
    return _finish(feat, thr, left, right, val, dleft, roots, 'gbdt', float(np.ravel(model._baseline_prediction)[0]))

def pack_forest(model):
    feat, thr, left, right, val, dleft, roots = [], [], [], [], [], [], []
    for t in model.estimators_:
        tr, off = t.tree_, len(feat)
        roots.append(off)
        v = tr.value[:, 0, :]
        p1 = v[:, 1] / np.maximum(v.sum(1), 1e-12)
        for i in range(tr.node_count):
            leaf = tr.children_left[i] == -1
            feat.append(-1 if leaf else int(tr.feature[i])); thr.append(float(tr.threshold[i]))
            left.append(-1 if leaf else off + int(tr.children_left[i])); right.append(-1 if leaf else off + int(tr.children_right[i]))
            val.append(float(p1[i])); dleft.append(True)
    return _finish(feat, thr, left, right, val, dleft, roots, 'forest', 0.0, f32=True)   # scikit-learn trees compare float32 inputs

PACKERS = {'LightGBM': pack_lightgbm, 'XGBoost': pack_xgboost, 'HistGradientBoosting': pack_hgb,
           'Random forest': pack_forest, 'Extra trees': pack_forest}

def fit_final(c):
    y = LAB[c].to_numpy(); k = ~np.isnan(y)
    return fit_model(make_final, X_ALL[k], y[k], X_counter=counter_rows(X_ALL))

t0 = time.time()
FINAL_MODELS = dict(zip(CROPS, joblib.Parallel(n_jobs=N_JOBS)(joblib.delayed(fit_final)(c) for c in CROPS)))
print(f'{len(FINAL_MODELS)} deployable crop models trained on all farmland cells ({time.time() - t0:.0f} s)')

packs = [PACKERS[FINAL_NAME](FINAL_MODELS[c]) for c in CROPS]
cat = {k: [] for k in ('feature', 'threshold', 'left', 'right', 'value', 'default_left', 'is_leaf', 'roots', 'tree_crop')}
off = 0
for ci, pk in enumerate(packs):
    for k in ('feature', 'threshold', 'value', 'default_left', 'is_leaf'):
        cat[k].append(pk[k])
    for k in ('left', 'right', 'roots'):
        cat[k].append(pk[k] + off)
    cat['tree_crop'].append(np.full(pk['roots'].size, ci, np.int16))
    off += pk['feature'].size
PFX = 'c_'
arrays = {PFX + k: np.concatenate(v) for k, v in cat.items()}
arrays[PFX + 'kind'] = np.array(packs[0]['kind'])
arrays[PFX + 'base'] = np.array([pk['base'] for pk in packs], np.float64)
arrays[PFX + 'input_f32'] = np.array(bool(packs[0]['f32']))
TRAIN_MIN, TRAIN_MAX = farm[FEATURES].min().to_numpy(), farm[FEATURES].max().to_numpy()
ENGINE_META = {
    'version': CFG['version'], 'algorithm': FINAL_NAME, 'params': {k: (v if isinstance(v, (int, float, str, type(None))) else str(v)) for k, v in FINAL_PARAMS.items()},
    'crops': CROPS, 'features': FEATURES, 'feature_nasa_parameter': FEATURE_PARAM,
    'modes': {'irrigated': {'prefix': PFX, 'features': FEATURES, 'irrigation_input': 'irr_share', 'irrigation_value': None},
              'rainfed': {'prefix': PFX, 'features': FEATURES, 'irrigation_input': 'irr_share', 'irrigation_value': 0.0}},
    'training_range': {'min': TRAIN_MIN.round(4).tolist(), 'max': TRAIN_MAX.round(4).tolist(),
                       'p01': P01.round(4).tolist(), 'p99': P99.round(4).tolist()},
    'suitability_threshold': CFG['suit_threshold'],
    'labels': f"MapSPAM 2020 share of harvested area: present >= {CFG['present_share']:.0%}, absent <= {CFG['absent_share']:.1%}",
    'training_cells': int(len(farm)), 'trained_utc': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'),
    'data_flags': DATA_FLAGS,
    'attribution': 'This data was provided by the International Food Policy Research Institute (IFPRI). IFPRI bears no responsibility '
                   'for the analyses or interpretations of the data presented here.',
}
NPZ_PATH = MODELS / 'crop_suitability_v2.npz'
np.savez_compressed(NPZ_PATH, meta_json=np.array(json.dumps(ENGINE_META)), **arrays)
eng = ce.CropSuitabilityEngine.load(NPZ_PATH)
n_trees, n_nodes = int(arrays[PFX + 'roots'].size), int(arrays[PFX + 'feature'].size)
print(f'packed model: {n_trees:,} trees, {n_nodes:,} nodes -> {NPZ_PATH.name} ({NPZ_PATH.stat().st_size / 1e6:.1f} MB)')

In [ ]:
def library_predict(X):
    return np.column_stack([FINAL_MODELS[c].predict_proba(X)[:, 1] for c in CROPS])

# fidelity: numpy engine vs the library models, on Egypt's yearly climates and a world sample, both modes
F_eg = annual.loc[annual.complete & annual.is_land, FEATURES].to_numpy()
F_w = farm[FEATURES].sample(3000, random_state=SEED).to_numpy()
FIDELITY = {}
for label, F in [('Egypt cell-years', F_eg), ('world farmland sample', F_w)]:
    Fc = ce.clip_features(F)
    for mode, lvl in [('irrigated', 1.0), ('rainfed', 0.0)]:
        t0 = time.time()
        a = eng.suitability(F, mode)
        t_np = time.time() - t0
        b = library_predict(np.column_stack([Fc, np.full(len(Fc), lvl)]))
        FIDELITY[f'{label}, {mode}'] = {'rows': len(F), 'max_abs_diff': float(np.nanmax(np.abs(a - b))),
                                        'same_suitable_call': float(np.mean((a >= 0.5) == (b >= 0.5))), 'numpy_seconds': round(t_np, 2)}
display(pd.DataFrame(FIDELITY).T)
assert max(v['max_abs_diff'] for v in FIDELITY.values()) < 1e-5, 'packed model does not reproduce the library model'
eng.attach_native(PFX, library_predict)   # bulk runs below use the (identical) library models for speed
print('numpy runtime reproduces the library models; bulk Egypt runs use the library path for speed.')

## 11. Egypt: held-out test, FAOSTAT check and comparison with v1
Three views of the same question: does the model rate the crops Egypt actually grows as suitable under irrigation?
* **Leave-Egypt-out (honest test):** the model never saw an Egyptian cell.
* **Deployed model (fit check):** trained with Egypt's 116 farmland cells. This shows what the app will display. It is not an independent test.
* **v1** (Kaggle crop-recommendation data from India, temperature and rainfall only), on the 13 crops both versions cover. v1 values come from the v1 notebook outputs (mean irrigated suitability over Nile farmland).

The national score is the farmland-weighted mean irrigated suitability over Egypt's farmland cells. FAOSTAT's "grown" list (≥ 10,000 ha harvested, mean of the last 10 years) is the truth. Absence in FAOSTAT is weaker evidence: markets and diet also decide what is grown.

In [ ]:
Xe = X_ALL[EGYPT].copy(); Xe[:, -1] = 1.0
DEPLOYED_EG = {c: FINAL_MODELS[c].predict_proba(Xe)[:, 1] for c in CROPS}
EG_DEPLOYED_TESTS, NAT_DEPLOYED = egypt_tests(DEPLOYED_EG)
NAT_HELDOUT = FINAL_NAT[FINAL_NAME]
V1_SCORE = {'rice': 0.973, 'maize': 0.950, 'beans': 0.829, 'banana': 0.268, 'cotton': 0.922, 'citrus': 0.984, 'temperate fruit': 0.896,
            'tropical fruit': 0.129, 'chickpea': 0.196, 'pigeon pea': 0.739, 'lentil': 0.990, 'coconut': 0.326, 'arabica coffee': 0.454}
V1_SOURCE = {'beans': 'kidney beans', 'citrus': 'orange', 'temperate fruit': 'apple', 'tropical fruit': 'mango', 'arabica coffee': 'coffee'}

def truth_scores(scores, crops=None):
    t = FAO_TRUTH.join(pd.Series(scores, name='score')).dropna(subset=['grown_in_egypt', 'score'])
    if crops is not None:
        t = t.loc[[c for c in crops if c in t.index]]
    g = t.grown_in_egypt == 1
    hit, fa = (t.score[g] >= CFG['suit_threshold']).mean(), (t.score[~g] >= CFG['suit_threshold']).mean()
    return {'crops': int(len(t)), 'grown': int(g.sum()), 'AUC': roc_auc_score(t.grown_in_egypt, t.score),
            'grown rated suitable': float(hit), 'not-grown rated suitable': float(fa), 'TSS': float(hit - fa)}

overlap = [c for c in V1_SCORE if c in FAO_TRUTH.dropna(subset=['grown_in_egypt']).index]
COMPARE = pd.DataFrame({'v1 (India data, T + rain)': truth_scores(V1_SCORE, overlap),
                        'v2 leave-Egypt-out (honest)': truth_scores(NAT_HELDOUT, overlap),
                        'v2 deployed (fit check)': truth_scores(NAT_DEPLOYED, overlap)}).T
ALL40 = pd.DataFrame({'v2 leave-Egypt-out (honest)': truth_scores(NAT_HELDOUT), 'v2 deployed (fit check)': truth_scores(NAT_DEPLOYED)}).T
print(f'On the {len(overlap)} crops covered by both versions:'); display(COMPARE.round(3))
print('On all v2 crops with a FAOSTAT label:'); display(ALL40.round(3))
display(pd.DataFrame({'leave-Egypt-out': FINAL_EGYPT[FINAL_NAME], 'deployed (fit check)': EG_DEPLOYED_TESTS}).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8.4), gridspec_kw={'width_ratios': [1, 1.05]})
ax = axes[0]
t = FAO_TRUTH.join(NAT_HELDOUT.rename('held')).join(NAT_DEPLOYED.rename('dep')).dropna(subset=['grown_in_egypt']).sort_values('dep')
yy_ = np.arange(len(t))
col = np.where(t.grown_in_egypt == 1, '#238b45', '#969696')
ax.barh(yy_, t.dep, color=col, alpha=.85, label='deployed model (trained with Egypt)')
ax.plot(t.held, yy_, 'o', mfc='white', mec='k', ms=5, label='leave-Egypt-out model (never saw Egypt)')
ax.axvline(CFG['suit_threshold'], c='k', ls='--', lw=1)
ax.set_yticks(yy_); ax.set_yticklabels(t.index, fontsize=7.5)
for lbl, g in zip(ax.get_yticklabels(), t.grown_in_egypt):
    lbl.set_color('#238b45' if g == 1 else '#636363')
ax.set_xlim(0, 1); ax.set_xlabel('irrigated suitability, farmland-weighted mean over Egypt')
ax.set_title('A. v2 on Egypt (green = grown at scale per FAOSTAT, grey = not grown)')
ax.legend(loc='lower right', frameon=True)
ax = axes[1]
o = sorted(overlap, key=lambda c: (-FAO_TRUTH.loc[c, 'grown_in_egypt'], c))
xx = np.arange(len(o)); w = 0.27
ax.bar(xx - w, [V1_SCORE[c] for c in o], w, color='#fdae6b', label=f"v1 (AUC {COMPARE.iloc[0]['AUC']:.2f}, TSS {COMPARE.iloc[0]['TSS']:.2f})")
ax.bar(xx, [NAT_HELDOUT[c] for c in o], w, color='#9ecae1', label=f"v2 leave-Egypt-out (AUC {COMPARE.iloc[1]['AUC']:.2f}, TSS {COMPARE.iloc[1]['TSS']:.2f})")
ax.bar(xx + w, [NAT_DEPLOYED[c] for c in o], w, color='#08519c', label=f"v2 deployed (AUC {COMPARE.iloc[2]['AUC']:.2f}, TSS {COMPARE.iloc[2]['TSS']:.2f})")
ax.axhline(CFG['suit_threshold'], c='k', ls='--', lw=1)
n_g = int((FAO_TRUTH.loc[o, 'grown_in_egypt'] == 1).sum())
ax.axvspan(n_g - .5, len(o) - .5, color='#f0f0f0', zorder=0)
ax.text(n_g / 2 - .5, 1.03, 'grown in Egypt', ha='center', fontsize=9, color='#238b45', fontweight='bold')
ax.text(n_g + (len(o) - n_g) / 2 - .5, 1.03, 'not grown in Egypt', ha='center', fontsize=9, color='#636363', fontweight='bold')
ax.set_xticks(xx); ax.set_xticklabels([f"{c}\n(v1: {V1_SOURCE[c]})" if c in V1_SOURCE else c for c in o], rotation=60, ha='right', fontsize=7.5)
ax.set_ylim(0, 1.32); ax.set_yticks(np.arange(0, 1.01, 0.2)); ax.set_ylabel('irrigated suitability in Egypt')
ax.set_title(f'B. v1 vs v2 on the {len(o)} crops both cover (FAOSTAT {FAO_PERIOD})')
ax.legend(loc='upper center', frameon=True, fontsize=7.5, ncol=2)
fig.suptitle('Figure 10. Does the model know what Egypt grows? Held-out test, deployed fit and the v1 comparison', fontweight='bold', y=1.0)
fig.tight_layout()
savefig(fig, 'fig10_egypt_faostat_v1_v2')

## 12. Egypt 2001 to 2025: suitability, trends and projections (F2, F3)
Each grid cell's monthly NASA POWER series (all six measurements) goes through the runtime engine. These are exactly the statistics the API computes for a live point: `ce.batch_reports` is `ce.point_report` for many points with one model call. For every cell and year we compute the 10 features.
* **Now:** the Theil-Sen trend-line value for 2025, which filters out single-year noise.
* **+5 and +10 years:** the same trend line extended, if current trends continue.
* **Uncertainty:** a moving-block bootstrap of the residual years (200 draws). Draws share the same years for all features and all cells, so a hot year stays a dry year and each draw is one coherent alternative history.

In [ ]:
param_by_cell = {}
for cid, g in wide.groupby('cell_id'):
    keys = g['time'].str.replace('-', '', regex=False)
    param_by_cell[cid] = {v: dict(zip(keys, g[v].astype(float))) for v in POWER_VARS}
ok_years = annual[annual.complete].groupby('cell_id').size()
land_cells = cells[cells.is_land & (cells.cell_id.map(ok_years).fillna(0) >= 10)].reset_index(drop=True)

def nearest_cell(lat, lon, table=land_cells):
    """Grid cell id whose centre is closest to (lat, lon); ids are never hard-coded."""
    d = (table['lat'] - lat) ** 2 + ((table['lon'] - lon) * np.cos(np.radians(lat))) ** 2
    return table['cell_id'].iloc[int(np.argmin(d.to_numpy()))]

RUN_KW = dict(base_year=CFG['base_year'], horizons=CFG['horizons'], n_boot=CFG['n_boot'], block_len=CFG['block_len'], seed=SEED,
              min_effect_suit=CFG['min_effect_suit'], min_effect_options=CFG['min_effect_options'])
t0 = time.time()
REPORTS = ce.batch_reports(eng, [(r.cell_id, r.lat, r.lon, param_by_cell[r.cell_id]) for r in land_cells.itertuples()],
                           include_series=False, **RUN_KW)
CAIRO = (30.04, 31.24)
cairo_id = nearest_cell(*CAIRO)
print(f'{len(REPORTS)} land cells processed in {time.time() - t0:.1f} s')
print(f'Example (Cairo, cell {cairo_id}):', REPORTS[cairo_id]['summary'])

In [ ]:
H5, H10 = (f'plus{h}' for h in CFG['horizons'])
DRIVERS = {'tmax_hottest': 'TX', 'rain_annual': 'RA', 'rh_mean': 'RH', 'soil_top_wetness': 'GT', 'soil_root_wetness': 'GR'}
cell_rows, crop_rows = [], []
for r in land_cells.itertuples():
    rep = REPORTS[r.cell_id]
    t, rr = rep['climate']['temperature_c'], rep['climate']['rainfall_mm_month']
    row = dict(cell_id=r.cell_id, lat=r.lat, lon=r.lon, in_egypt=r.in_egypt, nile_farmland=r.nile_farmland,
               farmland_ha=r.farmland_ha, irrigated_share=r.irrigated_share,
               T_now=t['now'], T_trend=t['trend_per_decade'], T_p=t['p_value'], T_conf=t['confidence'],
               R_now=rr['now'], R_trend=rr['trend_per_decade'], R_p=rr['p_value'], R_conf=rr['confidence'],
               in_training_range_now=rep['in_training_range']['now'], in_training_range_plus10=rep['in_training_range'][H10])
    for f, ab in DRIVERS.items():
        fe = rep['features'][f]
        row.update({f'{ab}_now': fe['now'], f'{ab}_trend': fe['trend_per_decade'], f'{ab}_p': fe['p_value'], f'{ab}_conf': fe['confidence']})
    for mode, mrep in rep['modes'].items():
        v = mrep['viability']
        row.update({f'{mode}_options_now': v['crop_options_now'], f'{mode}_n_suitable_now': v['n_suitable_now'],
                    f'{mode}_top3_now': v['top3_mean_now'], f'{mode}_trend': v['trend']['per_decade'],
                    f'{mode}_trend_p': v['trend']['p_value'], f'{mode}_trend_conf': v['trend']['confidence'],
                    f'{mode}_trend_dir': v['trend']['direction']})
        for h in (H5, H10):
            row.update({f'{mode}_options_{h}': v[h]['crop_options_point'], f'{mode}_delta_{h}': v[h]['delta_median'],
                        f'{mode}_delta_{h}_p10': v[h]['delta_p10'], f'{mode}_delta_{h}_p90': v[h]['delta_p90'],
                        f'{mode}_p_decline_{h}': v[h]['p_decline']})
        for c, e in mrep['crops'].items():
            cr = dict(cell_id=r.cell_id, lat=r.lat, lon=r.lon, in_egypt=r.in_egypt, nile_farmland=r.nile_farmland, mode=mode, crop=c,
                      now=e['now'], trend=e['trend']['per_decade'], trend_p=e['trend']['p_value'], trend_conf=e['trend']['confidence'],
                      trend_dir=e['trend']['direction'])
            for h in (H5, H10):
                cr.update({f'{h}': e[h]['point'], f'{h}_median': e[h]['median'], f'{h}_p10': e[h]['p10'], f'{h}_p90': e[h]['p90'],
                           f'{h}_p_cross': e[h]['p_cross'], f'{h}_cross': e[h]['cross']})
            crop_rows.append(cr)
    cell_rows.append(row)
cell_df = pd.DataFrame(cell_rows)
crop_long = pd.DataFrame(crop_rows)
crop_long['delta10'] = crop_long[H10] - crop_long['now']
eg_cells = cell_df[cell_df.in_egypt]
print(f"Egypt land cells: {len(eg_cells)} (farmland: {int(eg_cells.nile_farmland.sum())}) | warming with strong evidence: "
      f"{(eg_cells.T_conf == 'strong evidence').mean():.0%} | median warming {eg_cells.T_trend.median():.2f} C/decade | "
      f"climate inside the training range now: {eg_cells.in_training_range_now.mean():.0%}, at +10 years: {eg_cells.in_training_range_plus10.mean():.0%}")
display(eg_cells[['T_trend', 'TX_trend', 'RA_trend', 'RH_trend', 'GR_trend', 'irrigated_options_now', f'irrigated_options_{H10}',
                  f'irrigated_delta_{H10}', 'rainfed_options_now']].describe().round(3))

# sanity check of the rain-fed mode where it hardly rains (section 5: rain-fed counter-examples)
dry_eg = eg_cells[eg_cells.RA_now < 50]
RAINFED_CHECK = {'egypt_cells_below_50mm': int(len(dry_eg)),
                 'mean_rainfed_crops_suitable': float(dry_eg.rainfed_n_suitable_now.mean()),
                 'share_cells_no_rainfed_crop_suitable': float((dry_eg.rainfed_n_suitable_now == 0).mean()),
                 'mean_rainfed_crop_options': float(dry_eg.rainfed_options_now.mean()),
                 'mean_irrigated_crops_suitable': float(dry_eg.irrigated_n_suitable_now.mean()),
                 'cairo_rainfed_top5': [(x['crop'], x['suitability']) for x in REPORTS[cairo_id]['modes']['rainfed']['ranking']['now'][:5]],
                 'cairo_irrigated_top5': [(x['crop'], x['suitability']) for x in REPORTS[cairo_id]['modes']['irrigated']['ranking']['now'][:5]]}
print(f"Rain-fed check, {RAINFED_CHECK['egypt_cells_below_50mm']} Egyptian land cells with < 50 mm rain a year: "
      f"{RAINFED_CHECK['mean_rainfed_crops_suitable']:.2f} crops suitable on average in rain-fed mode "
      f"(no crop at all in {RAINFED_CHECK['share_cells_no_rainfed_crop_suitable']:.0%} of them) vs "
      f"{RAINFED_CHECK['mean_irrigated_crops_suitable']:.1f} in irrigated mode.")
print('Cairo rain-fed top 5 :', RAINFED_CHECK['cairo_rainfed_top5'])
print('Cairo irrigated top 5:', RAINFED_CHECK['cairo_irrigated_top5'])
for note in DATA_NOTES:
    print('Data note:', note)

### Maps

In [ ]:
LATS, LONS = np.sort(cells.lat.unique()), np.sort(cells.lon.unique())
dlat, dlon = float(np.diff(LATS).min()), float(np.diff(LONS).min())
lat_edges = np.r_[LATS - dlat / 2, LATS[-1] + dlat / 2]
lon_edges = np.r_[LONS - dlon / 2, LONS[-1] + dlon / 2]
LAT_I = {v: i for i, v in enumerate(LATS)}
LON_I = {v: i for i, v in enumerate(LONS)}
POLY = np.array(EGYPT_POLY + [EGYPT_POLY[0]])

def grid_of(df, col):
    G = np.full((len(LATS), len(LONS)), np.nan)
    for la, lo, v in zip(df['lat'], df['lon'], df[col]):
        G[LAT_I[la], LON_I[lo]] = np.nan if v is None else v
    return G

def draw_map(ax, df, col, cmap, title, cbar_label, norm=None, vmin=None, vmax=None, not_proven=None):
    if not_proven is not None:
        npg = grid_of(df.assign(_np=np.where(not_proven, 1.0, np.nan)), '_np')
        ax.pcolormesh(lon_edges, lat_edges, npg, cmap=ListedColormap(['#d9d9d9']), shading='flat')
        df = df.assign(**{col: np.where(not_proven, np.nan, df[col].astype(float))})
    m = ax.pcolormesh(lon_edges, lat_edges, grid_of(df, col), cmap=cmap, norm=norm, vmin=vmin, vmax=vmax, shading='flat')
    ax.plot(POLY[:, 0], POLY[:, 1], 'k-', lw=.9)
    f = df[df.nile_farmland]
    ax.plot(f['lon'], f['lat'], '.', c='k', ms=2.5)
    ax.set_xlim(lon_edges[0], lon_edges[-1]); ax.set_ylim(lat_edges[0], lat_edges[-1]); ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xticks(np.arange(26, 35.1, 2)); ax.set_yticks(np.arange(22, 32.1, 2))
    ax.set_xticklabels([f'{v:.0f}°E' for v in np.arange(26, 35.1, 2)], fontsize=7)
    ax.set_yticklabels([f'{v:.0f}°N' for v in np.arange(22, 32.1, 2)], fontsize=7)
    ax.grid(False)
    plt.colorbar(m, ax=ax, shrink=.75, label=cbar_label)
    return m

def sym(col):
    v = float(np.nanmax(np.abs(cell_df[col].astype(float))))
    return TwoSlopeNorm(0, -(v or 1e-6), v or 1e-6)

RAIN_STEP = bool(DATA_BREAKS.loc['rain_annual', 'possible data break'])
fig, axes = plt.subplots(2, 3, figsize=(17, 10.2))
specs = [('T_trend', 'T_conf', 'RdBu_r', 'A. T2M: mean temperature', 'deg C / decade'),
         ('TX_trend', 'TX_conf', 'RdBu_r', 'B. T2M_MAX: hottest monthly maximum', 'deg C / decade'),
         ('RA_trend', 'RA_conf', 'BrBG', 'C. PRECTOTCORR: annual rainfall' + (' *' if RAIN_STEP else ''), 'mm / year / decade'),
         ('RH_trend', 'RH_conf', 'BrBG', 'D. RH2M: relative humidity', '% points / decade'),
         ('GT_trend', 'GT_conf', 'BrBG', 'E. GWETTOP: surface soil wetness', 'wetness (0-1) / decade'),
         ('GR_trend', 'GR_conf', 'BrBG', 'F. GWETROOT: root-zone soil wetness', 'wetness (0-1) / decade')]
for ax, (col, conf, cmap, title, lab) in zip(axes.ravel(), specs):
    draw_map(ax, cell_df, col, cmap, title + ' (grey = not proven)', lab, norm=sym(col), not_proven=(cell_df[conf] == 'not proven').to_numpy())
fig.suptitle('Figure 11. The drivers: trends 2001 to 2025 in all six NASA POWER measurements (Theil-Sen per decade; dots = farmland cells)',
             fontweight='bold', y=1.0)
fig.tight_layout()
if RAIN_STEP:
    fig.text(0.5, -0.01, f"* Possible data break: Egypt's rainfall steps up from {int(DATA_BREAKS.loc['rain_annual', 'shift starts'])} "
             "(section 4). A trend line through a step overstates the long-term trend.", ha='center', fontsize=9, color='#cb181d')
savefig(fig, 'fig11_climate_drivers_six_measurements')

**Reading Figure 12.** *Crop options* is the sum of the 40 suitabilities, the "effective number of crops" the climate supports (the F2 viability index divides it by 40). It changes smoothly, which makes trends readable. Every crop adds a little, though, even a poorly suited one. So in a desert the rain-fed sum stays above zero although no crop is rated suitable. Panel D therefore counts the crops rated suitable (≥ 0.5) in rain-fed mode.

In [ ]:
NC = len(CROPS)
fig, axes = plt.subplots(1, 4, figsize=(21, 5.3))
draw_map(axes[0], cell_df, 'irrigated_options_now', 'viridis', 'A. Irrigated crop options now', f'effective number of crops (of {NC})', vmin=0)
draw_map(axes[1], cell_df, f'irrigated_delta_{H10}', 'PuOr', f'B. Change by +{CFG["horizons"][-1]} yrs (median, if trends continue)',
         'change in crop options', norm=sym(f'irrigated_delta_{H10}'))
draw_map(axes[2], cell_df, 'irrigated_trend', 'PuOr', 'C. Observed trend 2001 to 2025 (grey = not proven)', 'crop options / decade',
         norm=sym('irrigated_trend'), not_proven=(cell_df['irrigated_trend_conf'] == 'not proven').to_numpy())
draw_map(axes[3], cell_df.assign(_n=cell_df.rainfed_n_suitable_now.astype(float)), '_n', 'YlGn', 'D. Rain-fed: crops rated suitable now',
         f'crops with suitability >= {CFG["suit_threshold"]} (of {NC})', vmin=0)
fig.suptitle(f'Figure 12. Farmland viability (F2): how many of the {NC} crops the climate supports, and how that is changing',
             fontweight='bold', y=1.0)
fig.tight_layout()
savefig(fig, 'fig12_viability_maps')

In [ ]:
eg_irr = crop_long[(crop_long['mode'] == 'irrigated') & crop_long.in_egypt]
crossing = (eg_irr.assign(up=eg_irr[f'{H10}_cross'] == 'becomes suitable', down=eg_irr[f'{H10}_cross'] == 'loses suitability')
                  .groupby('crop')[['up', 'down', 'delta10']].agg({'up': 'sum', 'down': 'sum', 'delta10': 'mean'})).sort_values('delta10')
loser, gainer = crossing.index[0], crossing.index[-1]
fig = plt.figure(figsize=(18, 8.2))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.05])
for k, c in enumerate([gainer, loser]):
    ax = fig.add_subplot(gs[0, k])
    d = crop_long[(crop_long['mode'] == 'irrigated') & (crop_long.crop == c)]
    dv = float(np.nanmax(np.abs(d['delta10']))) or 1e-3
    draw_map(ax, d, 'delta10', 'PiYG', f'{"A" if k == 0 else "B"}. {c}: suitability change by +{CFG["horizons"][-1]} yrs',
             'change in suitability (0 to 1)', norm=TwoSlopeNorm(0, -dv, dv))
ax = fig.add_subplot(gs[0, 2])
cc = crossing[(crossing.up + crossing.down) > 0]
cc = cc if len(cc) else crossing
ax.barh(cc.index, cc['up'], color='#4d9221', label='cells becoming suitable')
ax.barh(cc.index, -cc['down'], color='#c51b7d', label='cells losing suitability')
ax.axvline(0, c='k', lw=.8)
ax.tick_params(axis='y', labelsize=7)
ax.set_xlabel(f'Egyptian land cells crossing the threshold by +{CFG["horizons"][-1]} years (p_cross >= 0.5)')
ax.set_title('C. Same warming, opposite outcomes (crops with any crossing)')
ax.legend(loc='lower right', frameon=True)
fig.suptitle("Figure 13. The statement's twist (R7): one warming signal, winners and losers side by side (irrigated mode)", fontweight='bold', y=1.0)
fig.tight_layout()
savefig(fig, 'fig13_winners_losers')
display(crossing.sort_values('delta10').round(3))

In [ ]:
farm_irr = crop_long[(crop_long['mode'] == 'irrigated') & crop_long.nile_farmland]
wts = cell_df.set_index('cell_id')['farmland_ha']
agg = farm_irr.assign(w=farm_irr.cell_id.map(wts)).groupby('crop').apply(
    lambda g: pd.Series({'now': np.average(g['now'], weights=g.w), 'plus10': np.average(g[H10], weights=g.w),
                         'p10': np.average(g[f'{H10}_p10'], weights=g.w), 'p90': np.average(g[f'{H10}_p90'], weights=g.w)}))
agg = agg.join(FAO_TRUTH['grown_in_egypt']).sort_values('now')
fig, ax = plt.subplots(figsize=(11, 10))
yy_ = np.arange(len(agg))
ax.hlines(yy_, agg.p10, agg.p90, color='#fdae6b', lw=5, alpha=.6, label=f'+{CFG["horizons"][-1]} years, 10 to 90 % range')
ax.plot(agg['now'], yy_, 'o', c='#08519c', ms=6, label=f'now ({CFG["base_year"]} trend value)')
ax.plot(agg['plus10'], yy_, 'D', c='#d94801', ms=5, label=f'+{CFG["horizons"][-1]} years if current trends continue')
ax.axvline(CFG['suit_threshold'], c='k', ls='--', lw=1)
ax.set_yticks(yy_); ax.set_yticklabels(agg.index, fontsize=8)
for lbl, g in zip(ax.get_yticklabels(), agg.grown_in_egypt):
    lbl.set_color('#238b45' if g == 1 else ('#636363' if g == 0 else '#000000'))
ax.set_xlim(0, 1); ax.set_xlabel('irrigated suitability, farmland-weighted mean over Egypt farmland')
ax.set_title('Green labels: grown at scale in Egypt (FAOSTAT); grey: not grown')
ax.legend(loc='lower right', frameon=True)
fig.suptitle(f'Figure 14. Egypt farmland, crop by crop (F3): now vs +{CFG["horizons"][-1]} years', fontweight='bold', y=0.995)
fig.tight_layout()
savefig(fig, 'fig14_egypt_farmland_crops_now_vs_plus10')

In [ ]:
cr = cells.set_index('cell_id').loc[cairo_id]
cairo = ce.point_report(eng, cr.lat, cr.lon, parameter=param_by_cell[cairo_id], include_series=True, **RUN_KW)
irr = cairo['modes']['irrigated']
top_now = [x['crop'] for x in irr['ranking']['now'][:6]]
movers = sorted(irr['crops'], key=lambda c: -abs((irr['crops'][c][H10]['median'] or 0) - (irr['crops'][c]['now'] or 0)))[:3]
pick = list(dict.fromkeys(top_now[:3] + movers))
yrs = cairo['climate']['temperature_c']['series']['years']
fig, axes = plt.subplots(1, 3, figsize=(19, 4.8), gridspec_kw={'width_ratios': [1, 1, 1.3]})
for ax, f, cc, lab in [(axes[0], 't_mean', '#d7301f', 'deg C'), (axes[1], 'soil_root_wetness', '#2171b5', 'root-zone soil wetness (0-1)')]:
    fe = cairo['features'][f]
    ax.plot(yrs, fe['series'], 'o-', c=cc, ms=3)
    xs = [CFG['base_year'] + h for h in CFG['horizons']]
    ax.errorbar(xs, [fe[f'plus{h}']['median'] for h in CFG['horizons']],
                yerr=[[fe[f'plus{h}']['median'] - fe[f'plus{h}']['p10'] for h in CFG['horizons']],
                      [fe[f'plus{h}']['p90'] - fe[f'plus{h}']['median'] for h in CFG['horizons']]], fmt='s', c='k', capsize=4,
                label='if current trends continue (median, 10 to 90 %)')
    ax.set_title(f"{'A' if f == 't_mean' else 'B'}. {fe['nasa_parameter']}: {fe['trend_per_decade']:+.3g} {fe['unit']}/decade ({fe['confidence']})")
    ax.set_ylabel(lab); ax.legend(frameon=True, fontsize=7)
ax = axes[2]
for c in pick:
    e = irr['crops'][c]
    line, = ax.plot(yrs, e['series'], '-', lw=1.6, label=c)
    xs = [CFG['base_year']] + [CFG['base_year'] + h for h in CFG['horizons']]
    ax.plot(xs, [e['now']] + [e[f'plus{h}']['median'] for h in CFG['horizons']], '--', c=line.get_color())
    ax.fill_between(xs, [e['now']] + [e[f'plus{h}']['p10'] for h in CFG['horizons']],
                    [e['now']] + [e[f'plus{h}']['p90'] for h in CFG['horizons']], color=line.get_color(), alpha=.15)
ax.axhline(CFG['suit_threshold'], c='grey', ls=':', lw=1)
ax.set_ylim(-.02, 1.02)
ax.set_title('C. Irrigated suitability: observed (solid), projected (dashed, 10 to 90 %)')
ax.legend(ncol=3, fontsize=8, frameon=True, loc='lower left')
fig.suptitle(f'Figure 15. Place to crops (F3) for the Cairo cell {cairo_id}', fontweight='bold', y=1.02)
fig.tight_layout()
savefig(fig, 'fig15_cairo_example')
print(cairo['summary'])

## 13. Decision views (F4): signals for planning, not instructions
Every row carries its evidence: suitability now, the projected value with its uncertainty, and the observed trend with its confidence label. In irrigated mode a desert cell scores as if water were available. The investor view is therefore split into **existing farmland** (MapSPAM farmland, where water is available today) and **new land**, which would need water development (desert reclamation such as Toshka or East Owainat).

In [ ]:
eg_crop = crop_long[crop_long.in_egypt & (crop_long['mode'] == 'irrigated')].copy()
eg_crop['land_type'] = np.where(eg_crop['nile_farmland'], 'existing farmland', 'new land (needs water development)')
invest = (eg_crop.sort_values([f'{H10}_p10', 'now'], ascending=False).groupby(['land_type', 'crop']).head(5)
                 .sort_values(['land_type', 'crop', f'{H10}_p10'], ascending=[True, True, False]))
invest = invest.assign(rank=invest.groupby(['land_type', 'crop']).cumcount() + 1)[
    ['land_type', 'crop', 'rank', 'cell_id', 'lat', 'lon', 'now', f'{H10}_median', f'{H10}_p10', f'{H10}_p90', 'trend_dir', 'trend_conf']]
lost = (eg_crop[eg_crop[f'{H10}_cross'] == 'loses suitability'].groupby('cell_id')['crop'].apply(lambda s: ', '.join(sorted(s))).rename('crops_at_risk'))
gained = (eg_crop[eg_crop[f'{H10}_cross'] == 'becomes suitable'].groupby('cell_id')['crop'].apply(lambda s: ', '.join(sorted(s))).rename('crops_gained'))
cols = ['cell_id', 'lat', 'lon', 'nile_farmland', 'irrigated_options_now', f'irrigated_options_{H10}', f'irrigated_delta_{H10}',
        f'irrigated_delta_{H10}_p10', f'irrigated_delta_{H10}_p90', f'irrigated_p_decline_{H10}', 'irrigated_trend', 'irrigated_trend_conf']
risk = (eg_cells[eg_cells[f'irrigated_p_decline_{H10}'] >= 0.8].sort_values(f'irrigated_delta_{H10}')[cols].head(20)
        .merge(lost, on='cell_id', how='left').merge(gained, on='cell_id', how='left'))
planner_decline = eg_cells[eg_cells['irrigated_trend_dir'] == 'losing'].sort_values('irrigated_trend')[cols].head(20)
planner_rise = eg_cells[eg_cells['irrigated_trend_dir'] == 'gaining'].sort_values('irrigated_trend', ascending=False)[cols].head(20)
print(f'Investor list: {len(invest)} rows | highest-risk cells: {len(risk)} | planner: {len(planner_decline)} declining, '
      f'{len(planner_rise)} rising (significant)')
display(risk.head(10).round(3))
display(invest[(invest.land_type == 'existing farmland') & invest.crop.isin(['wheat', 'maize', 'rice', 'cotton', 'citrus'])].round(3))

## 14. Export: artifacts for the API and the app

In [ ]:
def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    return o

META = {
    'engine_version': CFG['version'], 'generated_utc': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'),
    'region': 'egypt', 'source': 'NASA POWER monthly (MERRA-2 native grid), 2001 to 2025: T2M, T2M_MAX, PRECTOTCORR, RH2M, GWETTOP, GWETROOT',
    'grid': {'lat': [float(LATS[0]), float(LATS[-1]), dlat], 'lon': [float(LONS[0]), float(LONS[-1]), dlon]},
    'crops': CROPS, 'modes': ['irrigated', 'rainfed'], 'base_year': CFG['base_year'], 'horizons': CFG['horizons'],
    'features': {f: {'nasa_parameter': ce.FEATURE_INFO[f][0], 'description': ce.FEATURE_INFO[f][1], 'unit': ce.FEATURE_INFO[f][2]} for f in FEATURES},
    'model': {'algorithm': FINAL_NAME, 'training': f'MapSPAM 2020 crop presence on {len(farm):,} farmland cells worldwide', 'file': NPZ_PATH.name},
    'projection_label': ce.PROJECTION_LABEL, 'suitability_threshold': CFG['suit_threshold'],
    'suitability_classes': {'high': '>= 0.67', 'medium': '0.33 to 0.67', 'low': '< 0.33'},
    'confidence_levels': {'strong evidence': f"Mann-Kendall p < {CFG['p_strong']}", 'moderate evidence': f"p < {CFG['p_moderate']}",
                          'not proven': 'otherwise, or change smaller than the minimum effect'},
    'viability_definition': f'crop_options = sum of the {NC} crop suitabilities (effective number of crops the climate supports); '
                            f'viability_index = crop_options / {NC}; top3_mean = mean of the 3 best crops',
    'nile_farmland': 'Egyptian cells with >= 1,000 ha harvested in MapSPAM 2020 (where water is available today)',
    'note': 'Climate screening at about 55 km resolution. Irrigated mode assumes water is available; the nile_farmland flag marks where it is today.',
    'data_quality_notes': DATA_NOTES,
    'attribution': ENGINE_META['attribution'],
}
region = {'meta': META, 'cells': []}
for r in land_cells.itertuples():
    rep = REPORTS[r.cell_id]
    region['cells'].append({'cell_id': r.cell_id, 'lat': float(r.lat), 'lon': float(r.lon), 'in_egypt': bool(r.in_egypt),
                            'nile_farmland': bool(r.nile_farmland), 'farmland_ha': float(r.farmland_ha),
                            'climate': rep['climate'], 'features': rep['features'], 'in_training_range': rep['in_training_range'],
                            'modes': rep['modes'], 'summary': rep['summary']})
(OUTPUTS / 'egypt_crop_suitability.json').write_text(json.dumps(jsonable(region), separators=(',', ':')), encoding='utf-8')
(OUTPUTS / 'example_point_report_cairo.json').write_text(json.dumps(jsonable(cairo), indent=1), encoding='utf-8')
decisions = {'meta': {**META, 'wording': 'Signals for planning, not instructions. Every item shows its evidence and confidence.'},
             'investor_best_by_crop_plus10_irrigated': invest.to_dict('records'),
             'investor_highest_risk_of_losing_viability': risk.to_dict('records'),
             'planner_significant_declining_viability': planner_decline.to_dict('records'),
             'planner_significant_rising_viability': planner_rise.to_dict('records')}
(OUTPUTS / 'egypt_decisions.json').write_text(json.dumps(jsonable(decisions), indent=1), encoding='utf-8')
crop_long.to_csv(OUTPUTS / 'egypt_crop_suitability_long.csv', index=False)
cell_df.to_csv(OUTPUTS / 'egypt_cells_viability.csv', index=False)
invest.to_csv(OUTPUTS / 'decisions_investor_best_by_crop.csv', index=False)
risk.to_csv(OUTPUTS / 'decisions_highest_risk.csv', index=False)
pd.concat([planner_decline.assign(signal='declining'), planner_rise.assign(signal='rising')]).to_csv(OUTPUTS / 'decisions_planner.csv', index=False)
# evidence tables for the Methods page
ZOO_TAB.to_csv(OUTPUTS / 'model_zoo.csv')
pd.concat({n: t.assign(params=t.params.astype(str)) for n, t in TUNING.items()}).to_csv(OUTPUTS / 'tuning_candidates.csv')
FINAL_PER[FINAL_NAME].join(lab_summary).to_csv(OUTPUTS / 'per_crop_skill.csv')
IMP.to_csv(OUTPUTS / 'importance_by_nasa_parameter.csv')
FAO_TRUTH.join(NAT_HELDOUT.rename('score_leave_egypt_out')).join(NAT_DEPLOYED.rename('score_deployed')).join(
    pd.Series(V1_SCORE, name='score_v1')).to_csv(OUTPUTS / 'egypt_faostat_check.csv')
bundle_path = MODELS / 'crop_suitability_v2_sklearn.joblib'
joblib.dump({'models': FINAL_MODELS, 'algorithm': FINAL_NAME, 'params': FINAL_PARAMS, 'columns': COLS, 'crops': CROPS,
             'sklearn': sklearn.__version__}, bundle_path, compress=3)
print(f'library bundle (retraining/inspection; the app uses the npz): {bundle_path.stat().st_size / 1e6:.1f} MB')

METRICS = jsonable({
    'version': CFG['version'], 'config': CFG,
    'training_data': {'farmland_cells': int(len(farm)), 'spatial_blocks': int(len(np.unique(GROUPS))), 'egypt_farmland_cells': int(EGYPT.sum()),
                      'irrigated_farmland_cells': int(IRR_DOM.sum()), 'rainfed_farmland_cells': int(RF_DOM.sum()),
                      'feature_consistency_with_our_dataset_max_abs_diff': consistency['max |difference|'].astype(float).to_dict()},
    'data_quality': {'notes': DATA_NOTES, 'pettitt_egypt_farmland': DATA_BREAKS.to_dict('index')},
    'rainfed_counter_examples': {'counter_irr_share': CFG['counter_irr_share'], 'dry_rain_mm': CFG['dry_rain_mm'],
                                 'irrigated_copies_in_final_fit': int((X_ALL[:, -1] >= CFG['counter_irr_share']).sum()),
                                 'unfarmed_desert_cells': int(len(desert)),
                                 'false_alarm_metrics': [FA_COL, FA_DESERT_COL], 'egypt_rainfed_check': RAINFED_CHECK},
    'design_selection': DESIGN.to_dict('index'), 'selected_design': SELECTED_DESIGN, 'design_rank': DESIGN_RANK.to_dict(),
    'counter_example_ablation': COUNTER_ABLATION.astype(float).to_dict('index'),
    'model_zoo_spatial_cv': ZOO_TAB.to_dict('index'),
    'tuning': {n: {'best_params': {k: str(v) for k, v in BEST_PARAMS[n].items()},
                   'best': TUNING[n].drop(columns='params').iloc[0].to_dict(),
                   'zoo_default': TUNING[n][TUNING[n].candidate == 'zoo default'].drop(columns='params').iloc[0].to_dict()} for n in TOP3},
    'final_comparison_spatial_cv': FINAL_TAB.to_dict('index'), 'final_comparison_leave_egypt_out': EGYPT_TAB.to_dict('index'),
    'algorithm_decision': ALGO_DECISION,
    'selected_algorithm': FINAL_NAME, 'selected_params': {k: str(v) for k, v in FINAL_PARAMS.items()},
    'selected_cv_per_crop': FINAL_PER[FINAL_NAME].to_dict('index'), 'calibration_ece': CALIB,
    'importance_by_nasa_parameter_mean_auc_drop': imp_mean[::-1].to_dict(),
    'egypt_leave_out': FINAL_EGYPT[FINAL_NAME], 'egypt_deployed_fit_check': EG_DEPLOYED_TESTS,
    'faostat_v1_vs_v2_overlap': COMPARE.to_dict('index'), 'faostat_all_v2_crops': ALL40.to_dict('index'),
    'packed_model': {'file': NPZ_PATH.name, 'megabytes': NPZ_PATH.stat().st_size / 1e6, 'trees': n_trees, 'nodes': n_nodes},
    'numpy_runtime_fidelity': FIDELITY,
    'egypt_summary': {'land_cells': int(len(eg_cells)), 'farmland_cells': int(eg_cells.nile_farmland.sum()),
                      'median_warming_c_per_decade': float(eg_cells.T_trend.median()),
                      'share_cells_warming_strong_evidence': float((eg_cells.T_conf == 'strong evidence').mean()),
                      'median_change_irrigated_crop_options_plus10': float(eg_cells[f'irrigated_delta_{H10}'].median()),
                      'share_inside_training_range_now': float(eg_cells.in_training_range_now.mean())},
})
(OUTPUTS / 'metrics.json').write_text(json.dumps(METRICS, indent=1), encoding='utf-8')
shutil.rmtree(OUT / '__pycache__', ignore_errors=True)
print('Saved outputs:')
for p in sorted(OUT.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(OUT)}  ({p.stat().st_size / 1e6:.2f} MB)')

In [ ]:
fe_, fd_ = FINAL_EGYPT[FINAL_NAME], EG_DEPLOYED_TESTS
sel = FINAL_PER[FINAL_NAME].mean()
missed = [c for c in NAT_DEPLOYED.index if FAO_TRUTH.loc[c, 'grown_in_egypt'] == 1 and NAT_DEPLOYED[c] < CFG['suit_threshold']]
missed_lo = [c for c in NAT_HELDOUT.index if FAO_TRUTH.loc[c, 'grown_in_egypt'] == 1 and NAT_HELDOUT[c] < CFG['suit_threshold']]
params_txt = ', '.join(f'{k}={v}' for k, v in FINAL_PARAMS.items())
_unit = {f: ce.FEATURE_INFO[f][2] for f in FEATURES}
_cons = consistency['max |difference|'].astype(float)
cons_txt = '; '.join(f"{u.replace('deg C', '°C')}: {max(_cons[f] for f in FEATURES if _unit[f] == u):.2g}"
                     for u in dict.fromkeys(_unit.values()))
data_txt = ' '.join(DATA_NOTES) if DATA_NOTES else 'The Pettitt shift test found no possible data break in the six measurements over Egypt.'
card = f"""# Model card: CropShift Earth crop-suitability model ({CFG['version']})

**Task.** For any place on land, estimate for each of {NC} crops how typical the local climate is of places where that crop is grown today. The output is a suitability value from 0 to 1, in irrigated mode (water supplied) or rain-fed mode. It is applied to NASA POWER for now, +5 and +10 years if current trends continue.

**Inputs.** 10 climate features from all six NASA POWER measurements (T2M, T2M_MAX, PRECTOTCORR, RH2M, GWETTOP, GWETROOT): annual mean, coldest-month and warmest-month temperature; hottest monthly maximum; annual rainfall and its seasonality; mean humidity; surface and root-zone soil wetness (mean and driest month). One more input is the irrigation share: 1 in irrigated mode, 0 in rain-fed mode, anything in between from the app's slider.

**Training data.** IFPRI MapSPAM 2020 v2r2 harvested area (Harvard Dataverse doi:10.7910/DVN/SWPENT, CC BY 4.0), summed onto the NASA POWER grid (0.5 x 0.625 deg): {len(farm):,} farmland cells (>= 1,000 ha harvested) on every continent. Labels: crop present if it covers >= {CFG['present_share']:.0%} of the cell's harvested area, absent if <= {CFG['absent_share']:.1%}, otherwise left out. Features are the 2001 to 2025 averages from the NASA POWER archive. These match our Kaggle dataset for Egypt (largest difference per unit: {cons_txt}). *This data was provided by the International Food Policy Research Institute (IFPRI). IFPRI bears no responsibility for the analyses or interpretations of the data presented here.*

**Method.** One {FINAL_NAME} classifier per crop ({params_txt}), class-balanced weights, so 0.5 means the climate is as typical of cells that grow the crop as of cells that do not. The design (irrigation share as an input, share-based labels, rain-fed counter-examples) was chosen from {len(DESIGN_CAND) + 1} designs (one of them rain-fed only), plus an ablation without counter-examples (rain-fed false alarms on dry irrigated cells {COUNTER_ABLATION.loc[FA_COL, 'without']:.2f} without vs {COUNTER_ABLATION.loc[FA_COL, 'with counter-examples']:.2f} with; on unfarmed deserts {COUNTER_ABLATION.loc[FA_DESERT_COL, 'without']:.2f} vs {COUNTER_ABLATION.loc[FA_DESERT_COL, 'with counter-examples']:.2f}). Counter-examples, all with irrigation share 0 and the label "not grown": (1) every training cell that is >= {CFG['counter_irr_share']:.0%} irrigated, added once more, because MapSPAM shows no rain-fed farming there; (2) {len(desert):,} unfarmed desert cells (< {CFG['dry_rain_mm']} mm rain, < {CFG['farmland_min_ha']:,} ha farmed). Without them, rain-fed mode rated desert cells suitable. The algorithm was chosen from {len(FAMILIES)} classical ML families by 5-fold spatial block cross-validation (10 x 10 deg blocks). The three best families were tuned by random search on the class-balanced Brier score, under a size budget of {CFG['max_app_nodes']:,} tree nodes so the model runs in the app. {FINAL_NAME} had the best mean rank over the {len(METRIC_COLS)} cross-validation metrics (best on: {', '.join(ALGO_DECISION['metrics_won_by_selected']) or 'none outright'}); the runners-up are within {ALGO_DECISION['largest_gap_to_selected']['AUC']:.3f} AUC. The trees are packed into `{NPZ_PATH.name}` ({NPZ_PATH.stat().st_size / 1e6:.1f} MB, {n_trees:,} trees) and evaluated with numpy only. They reproduce the library model to {max(v['max_abs_diff'] for v in FIDELITY.values()):.1e}.

**Performance (held-out, spatial cross-validation, mean over {NC} crops).**
| Check | Result |
|---|---|
| AUC, all farmland / irrigated farmland (share = 1) / rain-fed farmland (share = 0) | {sel['AUC']:.3f} / {sel['AUC irrigated farmland']:.3f} / {sel['AUC rain-fed farmland']:.3f} |
| Worst crop AUC | {FINAL_PER[FINAL_NAME]['AUC'].min():.3f} ({FINAL_PER[FINAL_NAME]['AUC'].idxmin()}) |
| TSS at 0.5 / class-balanced Brier / calibration error (ECE) | {sel['TSS']:.3f} / {sel['Brier (balanced)']:.3f} / {CALIB[FINAL_NAME]:.3f} |
| Rain-fed false alarms: share of crops rated suitable in rain-fed mode on held-out farmland >= {CFG['counter_irr_share']:.0%} irrigated with < {CFG['dry_rain_mm']} mm rain | {sel[FA_COL]:.3f} |
| Rain-fed false alarms on held-out unfarmed deserts (< {CFG['dry_rain_mm']} mm rain, < {CFG['farmland_min_ha']:,} ha farmed) | {sel[FA_DESERT_COL]:.3f} |
| Egypt, cells with < 50 mm rain: crops suitable in rain-fed / irrigated mode (deployed model) | {RAINFED_CHECK['mean_rainfed_crops_suitable']:.2f} / {RAINFED_CHECK['mean_irrigated_crops_suitable']:.1f} |
| Leave-Egypt-out: AUC over Egypt's crop x cell pairs (MapSPAM) | {fe_['Egypt cells: AUC crop x cell (MapSPAM)']:.3f} |
| Leave-Egypt-out: FAOSTAT Egypt AUC / TSS (grown vs not grown) | {fe_['FAOSTAT AUC (grown vs not)']:.3f} / {fe_['FAOSTAT TSS']:.3f} |
| Deployed model (fit check, Egypt in training): FAOSTAT AUC / TSS | {fd_['FAOSTAT AUC (grown vs not)']:.3f} / {fd_['FAOSTAT TSS']:.3f} |
| v1 vs v2 on the {len(overlap)} shared crops: FAOSTAT AUC | v1 {COMPARE.iloc[0]['AUC']:.2f}, v2 held-out {COMPARE.iloc[1]['AUC']:.2f}, v2 deployed {COMPARE.iloc[2]['AUC']:.2f} |

**Most informative NASA measurements** (drop in held-out AUC when shuffled): {', '.join(f'{k} {v:.3f}' for k, v in imp_mean[::-1].items())}.

**Intended use.** Regional screening and storytelling (about 55 km cells): comparing places, crops and directions of change. Not for parcel-level investment decisions, yield prediction or agronomic advice.

**Limitations.**
* MapSPAM is itself a model. It spreads national statistics using crop-suitability layers (GAEZ), so part of its geography is climate-informed. We learn where crops *are* grown, which also reflects markets, policy, diet and water, not only climate.
* NASA's soil wetness comes from a land model without irrigation. In irrigated mode it describes the natural land, not the watered field. Irrigated mode assumes water exists. Use `nile_farmland` to see where it does today.
* Leave-Egypt-out, water-demanding crops that Egypt grows with Nile water score low ({', '.join(missed_lo) if missed_lo else 'none'}). Outside Egypt, irrigated deserts are mostly small groundwater farms. The deployed model includes Egypt's own farmland. Grown crops still below 0.5 in the deployed model: {', '.join(missed) if missed else 'none'}.
* Absence in FAOSTAT is weak evidence (economics decide too). Olive and date palm are not separate MapSPAM crops and are not covered.
* Data quality: {data_txt}
* Projections extend 2001 to 2025 Theil-Sen trends and are not climate forecasts. Future climates outside the training range are flagged (`in_training_range`). Mann-Kendall p-values are shown as confidence levels (strong / moderate / not proven), never as "the probability the trend is true". No serial-correlation correction.
"""
(OUT / 'MODEL_CARD.md').write_text(card, encoding='utf-8')
print(card)

## 15. Live demo: any point on Earth (needs Internet on Kaggle)
This is what `/crops?lat=..&lon=..` returns: a fresh NASA POWER request for all six measurements, the same engine and the same statistics. It uses the **numpy-only** runtime loaded from the `.npz`, exactly as the API would. It shows one point in Egypt and one far outside it, which is the F1 "done" criterion.

In [ ]:
eng_np = ce.CropSuitabilityEngine.load(NPZ_PATH)          # fresh engine: numpy evaluator, no library models attached
LIVE = {}
for name, (la, lo) in {'Cairo, Egypt': CAIRO, 'Nairobi, Kenya': (-1.29, 36.82)}.items():
    try:
        t0 = time.time()
        live = ce.point_report(eng_np, la, lo, start=2001, end=CFG['base_year'], **{k: v for k, v in RUN_KW.items() if k != 'base_year'})
        LIVE[name] = live
        print(f"{name} ({time.time() - t0:.1f} s incl. NASA POWER request): {live['summary']}")
        print('   irrigated now :', [(x['crop'], x['suitability']) for x in live['modes']['irrigated']['ranking']['now'][:5]])
        print(f"   irrigated +{CFG['horizons'][-1]}:", [(x['crop'], x['suitability']) for x in live['modes']['irrigated']['ranking'][H10][:5]])
        print('   rain-fed now  :', [(x['crop'], x['suitability']) for x in live['modes']['rainfed']['ranking']['now'][:5]])
    except Exception as e:
        print(f'{name}: live NASA POWER request skipped ({type(e).__name__}: {e}). Turn on Internet in the Kaggle notebook settings.')
t0 = time.time()
_ = ce.point_report(eng_np, cr.lat, cr.lon, parameter=param_by_cell[cairo_id], **RUN_KW)
print(f'numpy-only runtime, one full report (2 modes x {NC} crops x {CFG["n_boot"]} bootstrap draws): {time.time() - t0:.2f} s')
if 'Cairo, Egypt' in LIVE:   # consistency: live API path vs precomputed grid path for the same MERRA-2 cell
    a_ = LIVE['Cairo, Egypt']['modes']['irrigated']['crops']
    b_ = REPORTS[cairo_id]['modes']['irrigated']['crops']
    dmax = max(abs((a_[c]['now'] or 0) - (b_[c]['now'] or 0)) for c in CROPS)
    print(f'Live Cairo vs precomputed grid cell {cairo_id}: max difference in irrigated suitability = {dmax:.4f} '
          f"({'the API and the maps agree' if dmax < 0.01 else 'NASA POWER data version differs from our Kaggle snapshot'})")

In [ ]:
archive = shutil.make_archive(str(OUT.parent / 'cropshift_ml_v2'), 'zip', root_dir=OUT)
print('Download everything as one file:', archive, f'({Path(archive).stat().st_size / 1e6:.1f} MB)')
print(f'Total notebook run time: {(time.time() - T_START) / 60:.1f} min')

## 16. How Lane B plugs this in, and what we are honest about
**API (FastAPI) sketch.** The runtime needs only `numpy`. Copy `crop_engine_v2.py` and `models/crop_suitability_v2.npz` into the backend.
```python
import json, crop_engine_v2 as ce
ENG = ce.CropSuitabilityEngine.load("models/crop_suitability_v2.npz")
REGION = json.load(open("outputs/egypt_crop_suitability.json"))            # precomputed maps (F2/F3)

@app.get("/crops")
def crops(lat: float, lon: float, irrigation: float = 1.0):
    return ce.point_report(ENG, lat, lon, irrigation_level=irrigation)   # live, any point on Earth, all six NASA measurements
```
* `egypt_decisions.json` feeds the F4 lists as is. Field names stay compatible with v1 (`climate.temperature_c`, `climate.rainfall_mm_month`, `modes.irrigated/rainfed`, `viability`, `ranking`). v2 adds `features` (all 10 NASA-derived features with trends and projections) and `in_training_range`.
* The irrigation slider maps to `irrigation_level` (0 to 1) in irrigated mode; rain-fed mode always uses 0.
* `meta.data_quality_notes` lists measurements that the Pettitt test flagged as a possible data break over Egypt (section 4). The same flags travel inside the model file: every report for a point inside Egypt carries `data_quality_notes`, and the affected entries (`features.rain_annual`, `climate.rainfall_mm_month`, ...) carry a `data_quality_note`; the summary sentence then adds a caution. Show the note next to rainfall trends and rain-fed projections on the F2/F3 panels and on the Methods page (F5).

**Limitations (Methods page).** See `MODEL_CARD.md`. In short:
* Climate-only screening at about 55 km. No soil chemistry, salinity, water rights, pests or markets.
* Irrigated mode assumes water exists.
* MapSPAM is partly model-based.
* Projections are trend extrapolations, not forecasts.
* NASA POWER rainfall over Egypt shows a step around the mid-2010s (section 4), so Egypt's rainfall trend is not proven to be a climate signal.
* Leave-Egypt-out, Nile-watered summer crops are underrated. The deployed model learns them from Egypt's own farmland.